# A reproducible QSAR workflow, built from the board

**Histamine H3 receptor affinity against hERG liability, on compounds where both were measured in the same paper.**

Every number, plot and claim below is regenerated from the data each time this notebook runs. Nothing is hard-coded. Runtime → Run all reproduces the whole thing; the final section asserts that the reported metrics equal the ones recomputed from the stored predictions, and raises if they do not.

The workflow is the one drawn on the board `My reproducible QSAR WorkFlow.excalidraw` (SHA-256 `9df310c3598ed2830523026ab22a0010b14d3a999ceeba94b9192501cfb3f812`). The board's four hand-written **CHECKPOINT** markers are kept as this notebook's own checkpoints, so the two can be read side by side.

### What this is not

The target variable here is a difference of two binding affinities measured in cell-free and cell-based *in vitro* displacement assays. It is a cheminformatics quantity for ranking analogues inside one chemical series. It is **not** a clinical therapeutic index, it carries no dose and no exposure information, and nothing in it should be read as a statement about use in people. Section 17 says this again, at length, with the rest of the limitations.

In [ ]:
# Section 0. Reproducibility statement. Standard library only, so this runs before any install.
import hashlib, json, os, platform, sys, textwrap, time

SEED = 0                      # the one seed; every stochastic step below takes it
BOARD_SHA256 = "9df310c3598ed2830523026ab22a0010b14d3a999ceeba94b9192501cfb3f812"
BOARD_FILE = "My reproducible QSAR WorkFlow.excalidraw"
MODULE_SHA256 = {
    "substituent_constants": "4a6e751361f5b1f8988bdc7069f2046dc1333ce9e021a0faadd3cd010581b564",
    "qsar_core": "8a75259ef271c59465d5134ff9cb293a0e49dbd1c3d6fc9c2612f2a72ce54a2d",
    "series": "3950502abee88a93911442e341fbed831fe00880d68c7cdc6ed8ab92804d63c3",
    "curate": "b836111c67c651e3b600eff99fde21072532c4a1931525c340a2ce97d2a45ef0",
    "features": "4431917651fc37a5c75040cb87cc9bab1402aede4324a446915305069b00a36e",
    "splits": "9eed20e1368a3274f6550e48574398be4f7001ab122c064af21912b76049acc0",
    "schematics": "069bac6d95dcdaece93b2859208710eb396683a632a90b25a6e31be4285e3934",
    "whiteboard_sections": "ead691eee2dde674e7925e14ab758ad67f10effde650423c927df569ad366bf7"
}
DECLARED_PINS = {
    "numpy": "2.5.3",
    "pandas": "3.0.5",
    "scipy": "1.18.1",
    "rdkit": "2026.09.1",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.10.8",
    "optuna": "5.0.0",
    "shap": "0.54.1",
    "joblib": "1.5.2"
}

print("run at           ", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "UTC")
print("python           ", platform.python_version(), "on", platform.platform())
print("seed             ", SEED)
print("board            ", BOARD_FILE)
print("board sha256     ", BOARD_SHA256)

# The git commit, when the notebook is run inside a checkout. In Colab there is none, and
# the module hashes below pin the code instead - by content, which is stronger than a ref.
commit = None
try:
    import subprocess
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True,
                            timeout=10).stdout.strip() or None
except Exception:
    pass
print("git commit       ", commit or "not a git checkout (module hashes pin the code instead)")

print("\nCode written by this notebook, by SHA-256 of the source:")
for name, h in MODULE_SHA256.items():
    print(f"  tools/{name}.py".ljust(36), h)

print("\nDeclared library pins (resolved and compared in section 1):")
for k, v in DECLARED_PINS.items():
    print(f"  {k:<16} {v}")

## §1 Environment

Each pinned library is imported first and installed only if it is missing, so a version Colab already ships is **reported**, not silently overwritten or silently used. Versions are printed after the install step, never before it, and any difference from the declared pin is called out.

In [ ]:
# Section 1a. Install only what is missing, then report what is actually loaded.
import importlib, subprocess, sys

IMPORT_NAMES = {"scikit-learn": "sklearn"}
installed, preexisting, failed = [], [], []

for package, pin in DECLARED_PINS.items():
    mod = IMPORT_NAMES.get(package, package)
    try:
        importlib.import_module(mod)
        preexisting.append(package)
    except ImportError:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{package}=={pin}"],
                           check=True)
            installed.append(package)
        except Exception as exc:                      # reported, not swallowed
            failed.append((package, repr(exc)))

print("already present :", ", ".join(preexisting) or "none")
print("installed here  :", ", ".join(installed) or "none")
if failed:
    print("FAILED to install:")
    for p, e in failed:
        print("  ", p, e)

In [ ]:
# Section 1b. The code this notebook runs, written to tools/ so it can be imported, read and
# hashed. The cells below are generated from the module files in the repository by
# build_notebook.py; the hash check two cells down proves they still match.
import os, sys
os.makedirs("tools", exist_ok=True)
if "tools" not in sys.path:
    sys.path.insert(0, "tools")

In [ ]:
%%writefile tools/substituent_constants.py
"""Published substituent constants for the Hansch (Tier 1) and Free-Wilson (Tier 2) models.

These are NOT measurements made here and NOT predictions. They are reference values
transcribed from the standard compilations:

  Hansch, C.; Leo, A.; Taft, R. W. "A Survey of Hammett Substituent Constants and
  Resonance and Field Parameters." Chem. Rev. 1991, 91 (2), 165-195.
  doi:10.1021/cr00002a004
  Hansch, C.; Leo, A.; Hoekman, D. "Exploring QSAR: Hydrophobic, Electronic, and Steric
  Constants"; ACS Professional Reference Book; American Chemical Society, 1995.

Transcription is the weak link in any table like this, so the notebook does not take it on
trust. `validate_pi_against_experiment` re-derives pi from experimental log P by its own
definition, pi(X) = logP(C6H5-X) - logP(C6H6), using only measured log P values pulled from
ChEMBL, and reports the agreement. A substituent whose tabulated pi disagrees with the
experimental derivation is named in the output rather than used silently.

Columns:
  pi      Hansch hydrophobic substituent constant, from the benzene system
  sigma_m Hammett constant, meta
  sigma_p Hammett constant, para
  Es      Taft steric parameter (None where the compilation gives no standard value)
  MR      molar refractivity, scaled by 0.1 as is conventional in QSAR tables
"""
from __future__ import annotations

from dataclasses import dataclass

# SMARTS/SMILES of the substituent as it attaches, used to match R-groups by structure
# rather than by name. The attachment point is written as [*].
CONSTANTS: dict[str, dict[str, float | None | str]] = {
    "H":        {"smiles": "[*][H]",            "pi":  0.00, "sigma_m":  0.00, "sigma_p":  0.00, "Es":  0.00, "MR": 0.103},
    "CH3":      {"smiles": "[*]C",              "pi":  0.56, "sigma_m": -0.07, "sigma_p": -0.17, "Es": -1.24, "MR": 0.565},
    "C2H5":     {"smiles": "[*]CC",             "pi":  1.02, "sigma_m": -0.07, "sigma_p": -0.15, "Es": -1.31, "MR": 1.030},
    "n-C3H7":   {"smiles": "[*]CCC",            "pi":  1.55, "sigma_m": -0.07, "sigma_p": -0.13, "Es": -1.60, "MR": 1.496},
    "i-C3H7":   {"smiles": "[*]C(C)C",          "pi":  1.53, "sigma_m": -0.07, "sigma_p": -0.15, "Es": -1.71, "MR": 1.496},
    "t-C4H9":   {"smiles": "[*]C(C)(C)C",       "pi":  1.98, "sigma_m": -0.10, "sigma_p": -0.20, "Es": -2.78, "MR": 1.962},
    "C6H5":     {"smiles": "[*]c1ccccc1",       "pi":  1.96, "sigma_m":  0.06, "sigma_p": -0.01, "Es": -2.55, "MR": 2.536},
    "F":        {"smiles": "[*]F",              "pi":  0.14, "sigma_m":  0.34, "sigma_p":  0.06, "Es": -0.46, "MR": 0.092},
    "Cl":       {"smiles": "[*]Cl",             "pi":  0.71, "sigma_m":  0.37, "sigma_p":  0.23, "Es": -0.97, "MR": 0.603},
    "Br":       {"smiles": "[*]Br",             "pi":  0.86, "sigma_m":  0.39, "sigma_p":  0.23, "Es": -1.16, "MR": 0.888},
    "I":        {"smiles": "[*]I",              "pi":  1.12, "sigma_m":  0.35, "sigma_p":  0.18, "Es": -1.40, "MR": 1.839},
    "OH":       {"smiles": "[*]O",              "pi": -0.67, "sigma_m":  0.12, "sigma_p": -0.37, "Es": -0.55, "MR": 0.285},
    "OCH3":     {"smiles": "[*]OC",             "pi": -0.02, "sigma_m":  0.12, "sigma_p": -0.27, "Es": -0.55, "MR": 0.787},
    "OC2H5":    {"smiles": "[*]OCC",            "pi":  0.38, "sigma_m":  0.10, "sigma_p": -0.24, "Es": None,  "MR": 1.247},
    "OCF3":     {"smiles": "[*]OC(F)(F)F",      "pi":  1.04, "sigma_m":  0.38, "sigma_p":  0.35, "Es": None,  "MR": 0.786},
    "NH2":      {"smiles": "[*]N",              "pi": -1.23, "sigma_m": -0.16, "sigma_p": -0.66, "Es": -0.61, "MR": 0.542},
    "N(CH3)2":  {"smiles": "[*]N(C)C",          "pi":  0.18, "sigma_m": -0.15, "sigma_p": -0.83, "Es": None,  "MR": 1.555},
    "NO2":      {"smiles": "[*][N+](=O)[O-]",   "pi": -0.28, "sigma_m":  0.71, "sigma_p":  0.78, "Es": -2.52, "MR": 0.736},
    "CN":       {"smiles": "[*]C#N",            "pi": -0.57, "sigma_m":  0.56, "sigma_p":  0.66, "Es": -0.51, "MR": 0.633},
    "CF3":      {"smiles": "[*]C(F)(F)F",       "pi":  0.88, "sigma_m":  0.43, "sigma_p":  0.54, "Es": -2.40, "MR": 0.502},
    "CHO":      {"smiles": "[*]C=O",            "pi": -0.65, "sigma_m":  0.35, "sigma_p":  0.42, "Es": None,  "MR": 0.688},
    "COCH3":    {"smiles": "[*]C(C)=O",         "pi": -0.55, "sigma_m":  0.38, "sigma_p":  0.50, "Es": -1.24, "MR": 1.118},
    "COOH":     {"smiles": "[*]C(O)=O",         "pi": -0.32, "sigma_m":  0.37, "sigma_p":  0.45, "Es": None,  "MR": 0.693},
    "CONH2":    {"smiles": "[*]C(N)=O",         "pi": -1.49, "sigma_m":  0.28, "sigma_p":  0.36, "Es": None,  "MR": 0.981},
    "SCH3":     {"smiles": "[*]SC",             "pi":  0.61, "sigma_m":  0.15, "sigma_p":  0.00, "Es": None,  "MR": 1.382},
    "SO2NH2":   {"smiles": "[*]S(N)(=O)=O",     "pi": -1.82, "sigma_m":  0.53, "sigma_p":  0.57, "Es": None,  "MR": 1.228},
}

CITATION = (
    "Hansch, Leo & Taft, Chem. Rev. 1991, 91, 165-195 (doi:10.1021/cr00002a004); "
    "Hansch, Leo & Hoekman, Exploring QSAR, ACS, 1995."
)

SOURCE_TYPE = "literature_compilation"


@dataclass(frozen=True)
class Anchor:
    """A pair whose experimental log P difference defines pi for one substituent."""
    substituent: str
    parent_smiles: str = "c1ccccc1"
    probe_smiles: str = ""


def self_consistency_failures() -> list[str]:
    """Checks that hold by the definition of each scale, independent of any measurement."""
    bad = []
    h = CONSTANTS["H"]
    for name, field in (("pi", "pi"), ("sigma_m", "sigma_m"), ("sigma_p", "sigma_p"), ("Es", "Es")):
        if h[field] != 0.0:
            bad.append(f"H must have {name} = 0 by definition, table says {h[field]}")
    for key, row in CONSTANTS.items():
        if row["MR"] is None or float(row["MR"]) <= 0:
            bad.append(f"{key}: molar refractivity must be positive")
        es = row["Es"]
        if es is not None and float(es) > 0:
            bad.append(f"{key}: Taft Es is defined as <= 0 (H = 0), table says {es}")
    # Halogen ordering is a textbook consequence of size: F < Cl < Br < I in both pi and -Es.
    halogens = ["F", "Cl", "Br", "I"]
    pis = [float(CONSTANTS[h]["pi"]) for h in halogens]
    if pis != sorted(pis):
        bad.append(f"halogen pi should increase F<Cl<Br<I, table gives {pis}")
    eses = [-float(CONSTANTS[h]["Es"]) for h in halogens]
    if eses != sorted(eses):
        bad.append(f"halogen -Es should increase F<Cl<Br<I, table gives {eses}")
    return bad


def probe_smiles(substituent: str) -> str | None:
    """The monosubstituted benzene whose experimental log P gives pi for this substituent."""
    frag = CONSTANTS[substituent]["smiles"]
    if not isinstance(frag, str):
        return None
    tail = frag.replace("[*]", "", 1)
    if tail == "[H]":
        return "c1ccccc1"
    return "c1ccccc1" + tail if tail.startswith(("C", "O", "N", "S", "F", "I", "B", "[", "c")) else None

In [ ]:
%%writefile tools/qsar_core.py
"""The statistics a QSAR model has to report, written out rather than imported.

Every quantity here is computed from the data it is given. Nothing is cached, defaulted or
filled in. The functions are deliberately small so each reported number can be traced to the
lines that produced it, which is what the OECD principles and the Gramatica validation papers
ask for.

References for the external-validation metrics:
  Chirico, N.; Gramatica, P. "Real External Predictivity of QSAR Models: How To Evaluate It?
  Comparison of Different Validation Criteria and Proposal of Using the Concordance
  Correlation Coefficient." J. Chem. Inf. Model. 2011, 51 (9), 2320-2335.
  Consonni, V.; Ballabio, D.; Todeschini, R. "Comments on the Definition of the Q2
  Parameter for QSAR Validation." J. Chem. Inf. Model. 2009, 49 (7), 1669-1678.
  Roy, K. et al. r2m metrics, Chemom. Intell. Lab. Syst. 2012, 118, 200-210.
"""
from __future__ import annotations

import itertools
import math
from dataclasses import dataclass, field

import numpy as np


def _design(X: np.ndarray) -> np.ndarray:
    return np.column_stack([np.ones(len(X)), X])


@dataclass
class OLSFit:
    names: list[str]
    coef: np.ndarray            # intercept first
    se: np.ndarray
    t: np.ndarray
    p: np.ndarray
    n: int
    k: int                      # fitted terms, excluding the intercept
    r2: float
    r2_adj: float
    rmse: float
    s: float                    # standard error of the estimate
    f_stat: float
    f_p: float
    vif: np.ndarray             # one per term, excluding the intercept
    aicc: float
    fitted: np.ndarray
    residuals: np.ndarray
    leverage: np.ndarray
    h_star: float
    std_residuals: np.ndarray
    cov: np.ndarray            # covariance of the coefficients, intercept first

    def equation(self, response: str = "log(1/C)", dp: int = 3) -> str:
        parts = [f"{self.coef[0]:+.{dp}f}"]
        for name, c in zip(self.names, self.coef[1:]):
            parts.append(f"{c:+.{dp}f}·{name}")
        return f"{response} = " + " ".join(parts)


def _t_sf(t: float, df: int) -> float:
    """Two-sided p-value for Student's t, via the regularised incomplete beta function."""
    from scipy import stats
    return float(2.0 * stats.t.sf(abs(t), df))


def ols(X: np.ndarray, y: np.ndarray, names: list[str]) -> OLSFit:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    n, k = X.shape
    if n <= k + 1:
        raise ValueError(f"{n} compounds cannot support {k} fitted terms plus an intercept")
    A = _design(X)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    fitted = A @ coef
    resid = y - fitted
    df_res = n - k - 1
    sse = float(resid @ resid)
    sst = float(((y - y.mean()) ** 2).sum())
    mse = sse / df_res
    s = math.sqrt(mse)
    xtx_inv = np.linalg.pinv(A.T @ A)
    se = np.sqrt(np.clip(np.diag(xtx_inv) * mse, 0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(se > 0, coef / se, np.nan)
    p = np.array([_t_sf(ti, df_res) if np.isfinite(ti) else np.nan for ti in t])
    r2 = 1.0 - sse / sst if sst > 0 else float("nan")
    r2_adj = 1.0 - (1.0 - r2) * (n - 1) / df_res if df_res > 0 else float("nan")
    if k > 0 and sst > sse:
        f_stat = ((sst - sse) / k) / mse
        from scipy import stats
        f_p = float(stats.f.sf(f_stat, k, df_res))
    else:
        f_stat, f_p = float("nan"), float("nan")

    vif = np.full(k, np.nan)
    for j in range(k):
        others = np.delete(X, j, axis=1)
        if others.shape[1] == 0:
            vif[j] = 1.0
            continue
        Aj = _design(others)
        cj, *_ = np.linalg.lstsq(Aj, X[:, j], rcond=None)
        rj = X[:, j] - Aj @ cj
        ss_tot = float(((X[:, j] - X[:, j].mean()) ** 2).sum())
        r2j = 1.0 - float(rj @ rj) / ss_tot if ss_tot > 0 else 0.0
        vif[j] = 1.0 / (1.0 - r2j) if r2j < 1.0 else float("inf")

    params = k + 2  # terms + intercept + variance
    aic = n * math.log(sse / n) + 2 * params
    aicc = aic + (2 * params * (params + 1)) / (n - params - 1) if n - params - 1 > 0 else float("inf")

    hat = A @ xtx_inv @ A.T
    lev = np.clip(np.diag(hat), 0, None)
    h_star = 3.0 * (k + 1) / n
    std_resid = resid / s if s > 0 else resid * 0.0

    return OLSFit(names=list(names), coef=coef, se=se, t=t, p=p, n=n, k=k, r2=r2,
                  r2_adj=r2_adj, rmse=math.sqrt(sse / n), s=s, f_stat=f_stat, f_p=f_p,
                  vif=vif, aicc=aicc, fitted=fitted, residuals=resid, leverage=lev,
                  h_star=h_star, std_residuals=std_resid, cov=xtx_inv * mse)


def parabolic_optimum(fit: OLSFit, linear: str, square: str, alpha: float = 0.05) -> dict:
    """The turning point of a parabolic term, with a delta-method confidence interval.

    For y = ... + b*x + c*x^2 the optimum is x* = -b / (2c). Its uncertainty comes from the
    coefficient covariance by the delta method, with gradient (-1/(2c), b/(2c^2)). A positive
    c means the parabola opens upward, so the turning point is a minimum, not the optimum
    Hansch describes - that is reported rather than hidden, because it says the transport
    term did not behave.
    """
    from scipy import stats

    names = ["intercept"] + list(fit.names)
    try:
        ib, ic = names.index(linear), names.index(square)
    except ValueError:
        return {"available": False, "reason": f"{linear} and {square} are not both in the model"}
    b, c = float(fit.coef[ib]), float(fit.coef[ic])
    if c == 0:
        return {"available": False, "reason": "the squared term's coefficient is zero"}
    x_star = -b / (2 * c)
    g = np.zeros(len(names))
    g[ib] = -1.0 / (2 * c)
    g[ic] = b / (2 * c * c)
    var = float(g @ fit.cov @ g)
    se = math.sqrt(var) if var > 0 else float("nan")
    tcrit = float(stats.t.ppf(1 - alpha / 2, fit.n - fit.k - 1))
    return {"available": True, "optimum": x_star, "se": se,
            "ci_low": x_star - tcrit * se, "ci_high": x_star + tcrit * se,
            "opens": "downward (a maximum, as Hansch's transport term predicts)" if c < 0
                     else "upward (a minimum, not a transport optimum)",
            "b": b, "c": c, "alpha": alpha}


def q2_loo(X: np.ndarray, y: np.ndarray) -> tuple[float, np.ndarray]:
    """Leave-one-out Q2 against the mean of the full response, with the LOO predictions."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()
    n = len(y)
    preds = np.empty(n)
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        A = _design(X[mask])
        coef, *_ = np.linalg.lstsq(A, y[mask], rcond=None)
        preds[i] = float(np.concatenate(([1.0], X[i])) @ coef)
    press = float(((y - preds) ** 2).sum())
    sst = float(((y - y.mean()) ** 2).sum())
    return (1.0 - press / sst if sst > 0 else float("nan")), preds


def q2_lmo(X: np.ndarray, y: np.ndarray, fraction: float = 0.2, repeats: int = 100,
           seed: int = 0) -> dict:
    """Leave-many-out Q2: repeated random hold-out of a fraction of the training set."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()
    n = len(y)
    out = n if (m := max(1, int(round(fraction * n)))) >= n else m
    rng = np.random.default_rng(seed)
    scores = []
    sst = float(((y - y.mean()) ** 2).sum())
    for _ in range(repeats):
        idx = rng.permutation(n)[:out]
        mask = np.ones(n, dtype=bool)
        mask[idx] = False
        if mask.sum() <= X.shape[1] + 1:
            continue
        A = _design(X[mask])
        coef, *_ = np.linalg.lstsq(A, y[mask], rcond=None)
        pred = _design(X[~mask]) @ coef
        press = float(((y[~mask] - pred) ** 2).sum()) * n / out
        scores.append(1.0 - press / sst if sst > 0 else float("nan"))
    arr = np.array(scores, dtype=float)
    return {"mean": float(np.nanmean(arr)), "sd": float(np.nanstd(arr)),
            "min": float(np.nanmin(arr)), "max": float(np.nanmax(arr)),
            "repeats": int(len(arr)), "left_out_per_repeat": int(out)}


def y_randomisation(X: np.ndarray, y: np.ndarray, repeats: int = 100, seed: int = 0) -> dict:
    """Refit on shuffled responses. A real model's r2 and Q2 must sit above this cloud."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()
    rng = np.random.default_rng(seed)
    r2s, q2s = [], []
    for _ in range(repeats):
        ys = rng.permutation(y)
        r2s.append(ols(X, ys, [f"x{i}" for i in range(X.shape[1])]).r2)
        q2s.append(q2_loo(X, ys)[0])
    return {"r2_mean": float(np.mean(r2s)), "r2_max": float(np.max(r2s)),
            "q2_mean": float(np.mean(q2s)), "q2_max": float(np.max(q2s)),
            "repeats": repeats}


def ccc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Lin's concordance correlation coefficient, the criterion Chirico & Gramatica propose."""
    a = np.asarray(y_true, dtype=float).ravel()
    b = np.asarray(y_pred, dtype=float).ravel()
    n = len(a)
    if n < 2:
        return float("nan")
    va = float(((a - a.mean()) ** 2).sum()) / n
    vb = float(((b - b.mean()) ** 2).sum()) / n
    cov = float(((a - a.mean()) * (b - b.mean())).sum()) / n
    denom = va + vb + (a.mean() - b.mean()) ** 2
    return 2.0 * cov / denom if denom > 0 else float("nan")


def external_metrics(y_train: np.ndarray, y_ext: np.ndarray, pred_ext: np.ndarray) -> dict:
    """Q2F1, Q2F2, Q2F3, CCC and r2m, which disagree on purpose: report all of them."""
    y_train = np.asarray(y_train, dtype=float).ravel()
    y_ext = np.asarray(y_ext, dtype=float).ravel()
    pred = np.asarray(pred_ext, dtype=float).ravel()
    n_ext, n_tr = len(y_ext), len(y_train)
    press = float(((y_ext - pred) ** 2).sum())
    q2f1 = 1.0 - press / float(((y_ext - y_train.mean()) ** 2).sum())
    q2f2 = 1.0 - press / float(((y_ext - y_ext.mean()) ** 2).sum())
    sst_tr = float(((y_train - y_train.mean()) ** 2).sum())
    q2f3 = 1.0 - (press / n_ext) / (sst_tr / n_tr)
    r = float(np.corrcoef(y_ext, pred)[0, 1]) if n_ext > 1 else float("nan")
    r2 = r * r
    k = float((y_ext @ pred) / (pred @ pred)) if float(pred @ pred) > 0 else float("nan")
    r2_0 = 1.0 - float(((y_ext - k * pred) ** 2).sum()) / float(((y_ext - y_ext.mean()) ** 2).sum())
    r2m = r2 * (1.0 - math.sqrt(abs(r2 - r2_0))) if np.isfinite(r2_0) else float("nan")
    return {"n_external": n_ext, "q2_f1": q2f1, "q2_f2": q2f2, "q2_f3": q2f3,
            "ccc": ccc(y_ext, pred), "r2_ext": r2, "r2m": r2m,
            "rmse_ext": math.sqrt(press / n_ext),
            "mae_ext": float(np.abs(y_ext - pred).mean())}


@dataclass
class Subset:
    terms: tuple[int, ...]            # column indices, in the design's own order
    names: list[str]
    aicc: float
    n_variables: int                  # descriptors, counting a grouped pair as one
    n_coefficients: int               # fitted slopes, counting every column
    fit: OLSFit = field(repr=False)


def _units(p: int, groups: list[tuple[int, ...]] | None) -> list[tuple[int, ...]]:
    """Split the columns into selection units: a group is one unit, every other column its own.

    A unit is what the Topliss-Costello rule counts. The parabolic lipophilicity term
    (pi, pi^2) is one descriptor measured one way, so it is passed in as one group: it
    enters and leaves the model together and is counted once, while the two coefficients
    it fits are reported separately.
    """
    groups = [tuple(sorted(g)) for g in (groups or [])]
    seen: set[int] = set()
    for g in groups:
        if not g:
            raise ValueError("an empty group cannot be a descriptor")
        for j in g:
            if not 0 <= j < p:
                raise ValueError(f"group column {j} is outside the design's {p} columns")
            if j in seen:
                raise ValueError(f"column {j} appears in more than one group")
            seen.add(j)
    return sorted(groups + [(j,) for j in range(p) if j not in seen])


def best_subset_aicc(X: np.ndarray, y: np.ndarray, names: list[str],
                     max_terms: int | None = None,
                     compounds_per_term: int = 5,
                     groups: list[tuple[int, ...]] | None = None) -> tuple[Subset, list[Subset]]:
    """Exhaustive subsets up to the Topliss-Costello cap, ranked on AICc.

    Selecting on AICc and reporting Q2 keeps the criterion that chose the model separate
    from the criterion that judges it. The cap is Topliss & Costello's rule of at least
    `compounds_per_term` compounds for every fitted variable, where a variable is a
    selection unit: see `_units`. `groups` names the column sets that are one descriptor.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    n, p = X.shape
    units = _units(p, groups)
    cap = max(1, n // compounds_per_term)
    if max_terms is not None:
        cap = min(cap, max_terms)
    cap = min(cap, len(units))
    ranked: list[Subset] = []
    for size in range(1, cap + 1):
        for combo in itertools.combinations(units, size):
            cols = tuple(j for unit in combo for j in unit)
            sub = X[:, list(cols)]
            try:
                fit = ols(sub, y, [names[i] for i in cols])
            except (ValueError, np.linalg.LinAlgError):
                continue
            ranked.append(Subset(cols, [names[i] for i in cols], fit.aicc,
                                 n_variables=size, n_coefficients=len(cols), fit=fit))
    if not ranked:
        raise ValueError("no subset could be fitted")
    ranked.sort(key=lambda s: s.aicc)
    return ranked[0], ranked


def leverage_of(X_train: np.ndarray, X_query: np.ndarray) -> np.ndarray:
    """Hat leverage of query rows against a training design, for the applicability domain."""
    A = _design(np.asarray(X_train, dtype=float))
    inv = np.linalg.pinv(A.T @ A)
    Q = _design(np.asarray(X_query, dtype=float))
    return np.einsum("ij,jk,ik->i", Q, inv, Q)

In [ ]:
%%writefile tools/series.py
"""Congeneric series, R-group decomposition, and the Hansch descriptor block.

A Hansch equation is only meaningful inside a congeneric series: one parent structure with
substituents varied at defined positions. This finds those series in a set of molecules by
Bemis-Murcko scaffold, decomposes each series against its own scaffold, matches every
R-group to a published substituent constant by structure (not by name), and refuses the
compound if any of its R-groups has no tabulated constant - rather than estimating one.

For a series with several varying positions the descriptors are the sums over positions,
which is Hansch's own treatment of a polysubstituted series: sum(pi), sum(sigma), sum(Es),
sum(MR), plus sum(pi) squared for the parabolic lipophilicity term.
"""
from __future__ import annotations

import collections
from dataclasses import dataclass, field

from rdkit import Chem, RDLogger
from rdkit.Chem import rdRGroupDecomposition as rgd
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")

HANSCH_TERMS = ["sum_pi", "sum_pi_sq", "sum_sigma", "sum_Es", "sum_MR"]


def canonical_fragment(smiles_with_star: str) -> str | None:
    """Canonical SMILES of an R-group with its attachment point stripped to [*]."""
    s = smiles_with_star
    for i in range(1, 10):
        s = s.replace(f"[*:{i}]", "[*]")
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def constant_lookup(constants: dict) -> dict[str, str]:
    """Canonical fragment SMILES -> substituent name, so matching is structural."""
    out: dict[str, str] = {}
    for name, row in constants.items():
        key = canonical_fragment(str(row["smiles"]))
        if key is not None:
            out[key] = name
    return out


@dataclass
class Series:
    scaffold: str
    members: list[str] = field(default_factory=list)          # compound ids, in order
    r_groups: dict[str, dict[str, str]] = field(default_factory=dict)  # cmpd -> {R1: name}
    positions: list[str] = field(default_factory=list)
    unmatched_fragments: collections.Counter = field(default_factory=collections.Counter)
    dropped: dict[str, str] = field(default_factory=dict)      # cmpd -> reason

    @property
    def n(self) -> int:
        return len(self.members)

    def varying_positions(self) -> list[str]:
        return [p for p in self.positions
                if len({self.r_groups[c].get(p) for c in self.members}) > 1]


def find_series(records: list[dict], constants: dict, min_size: int = 8,
                smiles_key: str = "smiles", id_key: str = "compound_id") -> list[Series]:
    """Group by Murcko scaffold, decompose, and keep series with enough covered members."""
    lookup = constant_lookup(constants)
    by_scaffold: dict[str, list[tuple[str, Chem.Mol]]] = collections.defaultdict(list)
    for rec in records:
        mol = Chem.MolFromSmiles(rec[smiles_key])
        if mol is None:
            continue
        by_scaffold[MurckoScaffold.MurckoScaffoldSmiles(mol=mol)].append((rec[id_key], mol))

    series: list[Series] = []
    for scaffold, members in by_scaffold.items():
        if len(members) < min_size:
            continue
        core = Chem.MolFromSmiles(scaffold)
        if core is None:
            continue
        ids = [i for i, _ in members]
        mols = [m for _, m in members]
        try:
            rows, unmatched = rgd.RGroupDecompose([core], mols, asSmiles=True, asRows=True)
        except Exception:
            continue
        if unmatched:
            matched_ids = [i for k, i in enumerate(ids) if k not in set(unmatched)]
        else:
            matched_ids = ids
        if len(rows) != len(matched_ids):
            continue
        s = Series(scaffold=scaffold)
        positions = sorted({k for row in rows for k in row if k.startswith("R")})
        s.positions = positions
        for cmpd, row in zip(matched_ids, rows):
            named: dict[str, str] = {}
            missing = []
            for pos in positions:
                frag = row.get(pos)
                if frag is None:
                    named[pos] = "H"
                    continue
                key = canonical_fragment(frag)
                name = lookup.get(key or "")
                if name is None:
                    s.unmatched_fragments[key or frag] += 1
                    missing.append(f"{pos}={key or frag}")
                else:
                    named[pos] = name
            if missing:
                s.dropped[cmpd] = "no tabulated constant for " + "; ".join(missing)
                continue
            s.members.append(cmpd)
            s.r_groups[cmpd] = named
        if s.n >= min_size:
            series.append(s)
    series.sort(key=lambda s: -s.n)
    return series


def hansch_block(series: Series, constants: dict,
                 sigma_position: str = "sigma_p") -> tuple[list[str], list[list[float]], list[str]]:
    """Summed substituent constants for each member. Returns (compound ids, X, term names).

    Es is summed only over substituents for which the compilation gives a standard value; a
    series where any member's substituent has no Es is reported and the Es term is dropped
    rather than filled with a zero, because zero means hydrogen on that scale.
    """
    positions = series.varying_positions() or series.positions
    es_available = all(
        constants[series.r_groups[c][p]]["Es"] is not None
        for c in series.members for p in positions)
    terms = ["sum_pi", "sum_pi_sq", "sum_sigma", "sum_MR"]
    if es_available:
        terms.insert(3, "sum_Es")
    X: list[list[float]] = []
    for c in series.members:
        pi = sum(float(constants[series.r_groups[c][p]]["pi"]) for p in positions)
        sigma = sum(float(constants[series.r_groups[c][p]][sigma_position]) for p in positions)
        mr = sum(float(constants[series.r_groups[c][p]]["MR"]) for p in positions)
        row = [pi, pi * pi, sigma]
        if es_available:
            row.append(sum(float(constants[series.r_groups[c][p]]["Es"]) for p in positions))
        row.append(mr)
        X.append(row)
    return list(series.members), X, terms


def free_wilson_block(series: Series) -> tuple[list[str], list[list[float]], list[str]]:
    """Indicator variables, one per (position, substituent), hydrogen held as the reference."""
    positions = series.varying_positions()
    columns: list[tuple[str, str]] = []
    for p in positions:
        subs = sorted({series.r_groups[c][p] for c in series.members})
        for s in subs:
            if s == "H":
                continue
            columns.append((p, s))
    X = [[1.0 if series.r_groups[c][p] == s else 0.0 for p, s in columns]
         for c in series.members]
    names = [f"{p}:{s}" for p, s in columns]
    return list(series.members), X, names

In [ ]:
%%writefile tools/curate.py
"""Build the curated dataset and its provenance table from ChEMBL, filter by filter.

Two papers, chosen because each measures BOTH endpoints on the same compounds with one
assay per endpoint, so a therapeutic index can be formed without mixing assays:

  training  CHEMBL1821550  Bioorg Med Chem Lett 2011, doi:10.1016/j.bmcl.2011.07.006
            "Novel and highly potent histamine H3 receptor ligands. Part 1: withdrawing
            of hERG activity" - H3 Ki by [125I]iodoproxyfan displacement (CHEMBL1827363),
            hERG IC50 by [3H]dofetilide displacement (CHEMBL1827362).
  external  CHEMBL1155513  Bioorg Med Chem Lett 2010, doi:10.1016/j.bmcl.2010.04.045
            "In vitro studies on a class of quinoline containing histamine H3 antagonists"
            - H3 Ki by [3H]N-alpha-methylhistamine displacement (CHEMBL1120717), hERG Ki
            by [3H]dofetilide displacement in HEK293 cells (CHEMBL1120719).

Every value written here is a measurement reported in one of those papers and recorded by
ChEMBL. Nothing is predicted, imputed or read across, and the endpoint standard type is
never mixed within a set. Each filter reports rows in, rows out and its reason, and the
ChEMBL release the pull ran against is recorded alongside the row hash.
"""
from __future__ import annotations

import collections
import hashlib
import json
import math
import time
import urllib.parse
import urllib.request
from dataclasses import asdict, dataclass, field

BASE = "https://www.ebi.ac.uk/chembl/api/data"

# The four documents in ChEMBL that measure histamine H3 affinity and hERG affinity on the
# same molecules. Found, not assumed: `document_scan()` below reproduces the search, and no
# other document in the release pairs the two endpoints on ten compounds or more.
#
# CHEMBL1821550 is the training paper. The other three are external, and they are kept
# SEPARATE rather than pooled into one bag, because each uses a different readout:
#   CHEMBL1155513  H3 [3H]N-alpha-methylhistamine   hERG [3H]dofetilide displacement
#   CHEMBL2029265  H3 [3H]-(R)-alpha-methylhistamine hERG IonWorks patch clamp (FUNCTIONAL)
#   CHEMBL2375287  H3 [3H]histamine                 hERG [3H]astemizole displacement
# against the training paper's own H3 [125I]iodoproxyfan and hERG [3H]dofetilide. Metrics are
# reported per document and pooled, so the spread between laboratories is visible.
SETS = {
    "training": {
        "role": "training",
        "document": "CHEMBL1821550",
        "activity": {"target": "CHEMBL264", "standard_type": "Ki", "label": "H3"},
        "toxicity": {"target": "CHEMBL240", "standard_type": "IC50", "label": "hERG"},
    },
    "external_quinoline": {
        "role": "external",
        "document": "CHEMBL1155513",
        "activity": {"target": "CHEMBL264", "standard_type": "Ki", "label": "H3"},
        "toxicity": {"target": "CHEMBL240", "standard_type": "Ki", "label": "hERG"},
    },
    "external_ionworks": {
        "role": "external",
        "document": "CHEMBL2029265",
        "activity": {"target": "CHEMBL264", "standard_type": "Ki", "label": "H3"},
        "toxicity": {"target": "CHEMBL240", "standard_type": "IC50", "label": "hERG"},
    },
    "external_astemizole": {
        "role": "external",
        "document": "CHEMBL2375287",
        "activity": {"target": "CHEMBL264", "standard_type": "Ki", "label": "H3"},
        "toxicity": {"target": "CHEMBL240", "standard_type": "Ki", "label": "hERG"},
    },
}

H3_TARGET = "CHEMBL264"
HERG_TARGET = "CHEMBL240"

USER_AGENT = "drug-hansch-space-qsar-notebook (ChEMBL REST client)"


def _get(url: str, tries: int = 5) -> dict:
    """ChEMBL answers 500 to a request with no User-Agent, so always send one."""
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT,
                                                       "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=120) as r:
                return json.loads(r.read())
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("unreachable")


def paged(path: str, params: dict, limit: int = 1000) -> list[dict]:
    out: list[dict] = []
    offset = 0
    while True:
        q = dict(params, limit=limit, offset=offset, format="json")
        d = _get(f"{BASE}/{path}.json?" + urllib.parse.urlencode(q))
        key = next(k for k in d if k != "page_meta")
        batch = d[key]
        out.extend(batch)
        if len(batch) < limit:
            return out
        offset += limit


def chembl_release() -> dict:
    """The release the pull ran against. Without this a row hash pins nothing."""
    s = _get(f"{BASE}/status.json")
    return {"release": s.get("chembl_db_version"), "release_date": s.get("chembl_release_date"),
            "activities": s.get("activities"), "pulled_at": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                                                          time.gmtime())}


def document_citation(doc_id: str) -> dict:
    d = _get(f"{BASE}/document/{doc_id}.json")
    return {"document_chembl_id": doc_id, "doi": d.get("doi"), "pubmed_id": d.get("pubmed_id"),
            "journal": d.get("journal"), "year": d.get("year"), "title": d.get("title")}


# ---------------------------------------------------------------------------------------
# The filter ledger. Each entry is (label, reason, predicate); `apply_filters` runs them in
# order and returns the surviving rows plus a row-by-row account of what each one removed.
# ---------------------------------------------------------------------------------------

def _positive_value(r: dict) -> bool:
    v = r.get("standard_value")
    if v in (None, ""):
        return False
    try:
        return float(v) > 0
    except (TypeError, ValueError):
        return False


FILTERS: list[tuple[str, str, object]] = [
    ("assay_type == B", "a binding assay, so the number is an affinity and not a phenotype",
     lambda r: r.get("assay_type") == "B"),
    ("standard_relation == '='", "an exact value, not a censored '>' or '<' bound",
     lambda r: r.get("standard_relation") == "="),
    ("standard_units == 'nM'", "one concentration unit, so no unit conversion is needed",
     lambda r: r.get("standard_units") == "nM"),
    ("standard_value > 0", "a positive concentration, since the response is -log10 of it",
     _positive_value),
    ("data_validity_comment empty", "ChEMBL has not flagged the value as suspect",
     lambda r: not r.get("data_validity_comment")),
    ("pchembl_value present", "ChEMBL derived a p-value, which it declines to do for a "
                              "value it distrusts",
     lambda r: r.get("pchembl_value") not in (None, "")),
    ("canonical_smiles present", "there is a structure to compute descriptors from",
     lambda r: bool(r.get("canonical_smiles"))),
]


def apply_filters(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    ledger = []
    kept = list(rows)
    for label, reason, keep in FILTERS:
        before = len(kept)
        kept = [r for r in kept if keep(r)]
        ledger.append({"filter": label, "reason": reason, "rows_in": before,
                       "rows_out": len(kept), "dropped": before - len(kept)})
    return kept, ledger


@dataclass
class Record:
    set_name: str
    compound_id: str
    smiles: str
    endpoint: str
    target_chembl_id: str
    standard_type: str
    value_nM: float
    p_value: float
    unit: str
    source_type: str
    citation: str
    assay_chembl_id: str
    assay_description: str
    potential_duplicate: int = 0


@dataclass
class SetReport:
    name: str
    document: str
    rows_in_document: int
    ledger: list[dict] = field(default_factory=list)
    surviving_pairs: dict = field(default_factory=dict)
    endpoints: dict = field(default_factory=dict)
    duplicate_groups_collapsed: int = 0
    duplicate_flag_rows: dict = field(default_factory=dict)
    counts: dict = field(default_factory=dict)


def p_from_nM(value_nM: float) -> float:
    """-log10 of the molar concentration. The only arithmetic applied to a measurement."""
    return -math.log10(value_nM * 1e-9)


def collect(set_name: str, spec: dict) -> tuple[list[Record], dict, SetReport]:
    doc = spec["document"]
    cite = document_citation(doc)
    citation = f"doi:{cite['doi']}" if cite["doi"] else f"pmid:{cite['pubmed_id']}"
    raw = paged("activity", {"document_chembl_id": doc})
    kept, ledger = apply_filters(raw)
    report = SetReport(name=set_name, document=doc, rows_in_document=len(raw), ledger=ledger)
    report.surviving_pairs = {f"{t}/{st}": c for (t, st), c in sorted(
        collections.Counter((r["target_chembl_id"], r["standard_type"]) for r in kept).items())}

    records: list[Record] = []
    for role in ("activity", "toxicity"):
        want = spec[role]
        hits = [r for r in kept if r["target_chembl_id"] == want["target"]
                and r["standard_type"] == want["standard_type"]]
        assays = sorted({r["assay_chembl_id"] for r in hits})
        if len(assays) != 1:
            raise SystemExit(
                f"{set_name}/{role}: expected one assay, found {len(assays)}: {assays}")

        # potential_duplicate is ChEMBL's note that the same measurement appears elsewhere in
        # the database. It is recorded, not acted on: the key that matters for this dataset is
        # (compound, target, standard_type) inside one document, and that is deduplicated
        # explicitly below. Dropping every flagged row would delete measurements that are the
        # only row for their compound.
        flagged = sum(1 for r in hits if r.get("potential_duplicate"))
        groups: dict[str, list[dict]] = collections.defaultdict(list)
        for r in hits:
            groups[r["molecule_chembl_id"]].append(r)
        collapsed, disagreeing = 0, []
        chosen = []
        for mol, rs in groups.items():
            if len(rs) > 1:
                collapsed += 1
                vals = sorted({round(float(x["standard_value"]), 6) for x in rs})
                if len(vals) > 1:
                    disagreeing.append({"compound_id": mol, "values_nM": vals})
                rs = sorted(rs, key=lambda x: float(x["standard_value"]))
                rs = [rs[len(rs) // 2]]          # median row, so no value is invented
            chosen.append(rs[0])
        report.duplicate_groups_collapsed += collapsed
        report.duplicate_flag_rows[role] = flagged
        report.endpoints[role] = {
            "target": want["target"], "standard_type": want["standard_type"],
            "label": want["label"], "assay_chembl_id": assays[0],
            "assay_description": (hits[0].get("assay_description") or "") if hits else "",
            "rows": len(hits), "compounds": len(chosen),
            "rows_carrying_potential_duplicate": flagged,
            "compounds_with_several_rows": collapsed,
            "disagreeing_duplicates": disagreeing,
        }
        for r in chosen:
            v = float(r["standard_value"])
            records.append(Record(
                set_name=set_name, compound_id=r["molecule_chembl_id"],
                smiles=r["canonical_smiles"], endpoint=want["label"],
                target_chembl_id=want["target"], standard_type=want["standard_type"],
                value_nM=v, p_value=p_from_nM(v), unit="nM",
                source_type="experimental", citation=citation,
                assay_chembl_id=r["assay_chembl_id"],
                assay_description=r["assay_description"] or "",
                potential_duplicate=int(bool(r.get("potential_duplicate")))))

    by_cmpd: dict[str, dict] = {}
    for r in records:
        by_cmpd.setdefault(r.compound_id, {})[r.endpoint] = r
    labels = {spec[role]["label"] for role in ("activity", "toxicity")}
    report.counts = {
        "activity_measurements": sum(1 for r in records
                                     if r.endpoint == spec["activity"]["label"]),
        "toxicity_measurements": sum(1 for r in records
                                     if r.endpoint == spec["toxicity"]["label"]),
        "compounds_with_both": sum(1 for v in by_cmpd.values() if len(v) == len(labels)),
        "compounds_with_activity_only": sum(
            1 for v in by_cmpd.values() if set(v) == {spec["activity"]["label"]}),
        "compounds_with_toxicity_only": sum(
            1 for v in by_cmpd.values() if set(v) == {spec["toxicity"]["label"]}),
    }
    return records, cite, report


def document_scan(min_paired: int = 10) -> list[dict]:
    """Every ChEMBL document measuring BOTH H3 and hERG affinity on the same molecules.

    This is the search that chose the sets above, written so it can be re-run rather than
    taken on trust. It pages every binding activity for the two targets - about 5k rows for
    H3 and 12k for hERG in ChEMBL_37 - and intersects the molecule sets per document.
    """
    def by_document(target: str) -> dict[str, set[str]]:
        rows = paged("activity", {
            "target_chembl_id": target, "assay_type": "B", "standard_units": "nM",
            "standard_relation": "=",
            "only": "document_chembl_id,molecule_chembl_id,standard_type,pchembl_value"})
        out: dict[str, set[str]] = collections.defaultdict(set)
        for r in rows:
            if r.get("standard_type") in ("Ki", "IC50") and r.get("pchembl_value"):
                out[r["document_chembl_id"]].add(r["molecule_chembl_id"])
        return out

    h3, herg = by_document(H3_TARGET), by_document(HERG_TARGET)
    found = []
    for doc in set(h3) & set(herg):
        both = h3[doc] & herg[doc]
        if len(both) >= min_paired:
            found.append({"document": doc, "paired": len(both),
                          "h3_compounds": len(h3[doc]), "herg_compounds": len(herg[doc])})
    found.sort(key=lambda d: -d["paired"])
    return found


def build() -> dict:
    out = {"chembl": chembl_release(), "sets": {}, "citations": {}, "records": [],
           "reports": {}}
    for name, spec in SETS.items():
        recs, cite, report = collect(name, spec)
        out["citations"][name] = cite
        out["records"].extend(asdict(r) for r in recs)
        out["reports"][name] = asdict(report)
        out["sets"][name] = dict(report.counts, document=spec["document"],
                                 role=spec["role"])
    return out


def table(data: dict) -> list[dict]:
    """One row per compound: both measurements, the index, and where each came from."""
    rows = []
    for name, spec in SETS.items():
        act_label, tox_label = spec["activity"]["label"], spec["toxicity"]["label"]
        recs = [r for r in data["records"] if r["set_name"] == name]
        by: dict[str, dict] = {}
        for r in recs:
            by.setdefault(r["compound_id"], {})[r["endpoint"]] = r
        for cmpd, pair in sorted(by.items()):
            act, tox = pair.get(act_label), pair.get(tox_label)
            rows.append({
                "set": name,
                "role": spec["role"],
                "document": spec["document"],
                "compound_id": cmpd,
                "smiles": (act or tox)["smiles"],
                "p_activity_H3": act["p_value"] if act else None,
                "activity_standard_type": act["standard_type"] if act else None,
                "activity_value_nM": act["value_nM"] if act else None,
                "p_toxicity_hERG": tox["p_value"] if tox else None,
                "toxicity_standard_type": tox["standard_type"] if tox else None,
                "toxicity_value_nM": tox["value_nM"] if tox else None,
                "therapeutic_index": (act["p_value"] - tox["p_value"]) if (act and tox) else None,
                "endpoint_source_type": "experimental",
                "citation": (act or tox)["citation"],
                "activity_assay": act["assay_chembl_id"] if act else None,
                "toxicity_assay": tox["assay_chembl_id"] if tox else None,
                "missing": ",".join(
                    e for e, present in (("p_activity_H3", act), ("p_toxicity_hERG", tox))
                    if not present) or None,
            })
    return rows


def sha256_of_rows(rows: list[dict], columns: list[str]) -> str:
    h = hashlib.sha256()
    for r in rows:
        h.update(("|".join("" if r.get(c) is None else str(r[c]) for c in columns) + "\n").encode())
    return h.hexdigest()


if __name__ == "__main__":
    import csv
    import sys

    dest = sys.argv[1]
    data = build()
    rows = table(data)
    cols = list(rows[0])
    with open(dest, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)
    json.dump({"chembl": data["chembl"], "citations": data["citations"],
               "reports": data["reports"]},
              open(dest.replace(".csv", "_provenance.json"), "w"), indent=1)
    print("ChEMBL", data["chembl"]["release"], "released", data["chembl"]["release_date"])
    for name, s in data["sets"].items():
        print(f"{name}: {s}")
    print(f"rows {len(rows)} -> {dest}")
    print("sha256", sha256_of_rows(rows, cols))

In [ ]:
%%writefile tools/features.py
"""Structure standardisation, computed descriptors, property rules, cliffs, and selection.

Everything here is *computed*, not measured, and is tagged that way so a reader can never
mistake a Crippen log P for a shake-flask one. The Hansch block in `series.py` uses
tabulated experimental substituent constants instead; these descriptors serve Tiers 3 and 4
only, as the plan says.
"""
from __future__ import annotations

import collections
import math
from dataclasses import dataclass, field

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Crippen, Descriptors, QED, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

SOURCE_TYPE = "computed"

# One curated block rather than every RDKit descriptor: each of these is interpretable, and a
# wall of 200 correlated columns would make the genetic algorithm's job meaningless.
DESCRIPTORS: dict[str, object] = {
    "cLogP": Crippen.MolLogP,
    "cMR": Crippen.MolMR,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "MolWt": Descriptors.MolWt,
    "HeavyAtoms": lambda m: float(m.GetNumHeavyAtoms()),
    "HBD": rdMolDescriptors.CalcNumHBD,
    "HBA": rdMolDescriptors.CalcNumHBA,
    "RotBonds": rdMolDescriptors.CalcNumRotatableBonds,
    "Rings": rdMolDescriptors.CalcNumRings,
    "AromaticRings": rdMolDescriptors.CalcNumAromaticRings,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "Heteroatoms": lambda m: float(rdMolDescriptors.CalcNumHeteroatoms(m)),
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "BalabanJ": Descriptors.BalabanJ,
    "BertzCT": Descriptors.BertzCT,
    "HallKierAlpha": rdMolDescriptors.CalcHallKierAlpha,
    "Kappa1": rdMolDescriptors.CalcKappa1,
    "Kappa2": rdMolDescriptors.CalcKappa2,
    "Kappa3": rdMolDescriptors.CalcKappa3,
    "Chi0v": rdMolDescriptors.CalcChi0v,
    "Chi1v": rdMolDescriptors.CalcChi1v,
    "Chi2v": rdMolDescriptors.CalcChi2v,
    "Chi3v": rdMolDescriptors.CalcChi3v,
    "QED": QED.qed,
}

LIPOPHILICITY_PAIR = ("cLogP", "cLogP_sq")


def descriptor_names(parabolic: bool = True) -> list[str]:
    names = list(DESCRIPTORS)
    if parabolic:
        names.insert(names.index("cLogP") + 1, "cLogP_sq")
    return names


def descriptor_block(smiles: list[str], parabolic: bool = True
                     ) -> tuple[np.ndarray, list[str], list[int]]:
    """Returns (X, names, indices of the rows that could be computed)."""
    names = descriptor_names(parabolic)
    rows, ok = [], []
    for i, smi in enumerate(smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        vals = {k: float(f(mol)) for k, f in DESCRIPTORS.items()}
        if parabolic:
            vals["cLogP_sq"] = vals["cLogP"] ** 2
        row = [vals[n] for n in names]
        if not all(math.isfinite(v) for v in row):
            continue
        rows.append(row)
        ok.append(i)
    return np.asarray(rows, dtype=float), names, ok


def lipophilicity_group(names: list[str]) -> list[tuple[int, ...]]:
    """The (cLogP, cLogP^2) columns as one selection unit, per the project's logP rule."""
    if all(n in names for n in LIPOPHILICITY_PAIR):
        return [tuple(names.index(n) for n in LIPOPHILICITY_PAIR)]
    return []


# ---------------------------------------------------------------------------------------
# Standardisation
# ---------------------------------------------------------------------------------------

@dataclass
class Standardised:
    compound_id: str
    before: str
    after: str
    inchikey: str
    changed: bool
    note: str = ""


def standardise(records: list[dict], smiles_key: str = "smiles",
                id_key: str = "compound_id") -> tuple[list[Standardised], dict]:
    """Largest fragment, neutralise, canonical tautomer - counted before and after."""
    chooser = rdMolStandardize.LargestFragmentChooser()
    uncharger = rdMolStandardize.Uncharger()
    tauto = rdMolStandardize.TautomerEnumerator()
    out, stats = [], collections.Counter()
    for rec in records:
        smi = rec[smiles_key]
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            out.append(Standardised(rec[id_key], smi, "", "", False, "unparseable SMILES"))
            stats["unparseable"] += 1
            continue
        steps = []
        n_frags = len(Chem.GetMolFrags(mol))
        if n_frags > 1:
            mol = chooser.choose(mol)
            steps.append(f"largest of {n_frags} fragments")
        charged = any(a.GetFormalCharge() for a in mol.GetAtoms())
        if charged:
            neutral = uncharger.uncharge(mol)
            if Chem.MolToSmiles(neutral) != Chem.MolToSmiles(mol):
                mol = neutral
                steps.append("neutralised")
        canon = tauto.Canonicalize(mol)
        if Chem.MolToSmiles(canon) != Chem.MolToSmiles(mol):
            mol = canon
            steps.append("canonical tautomer")
        after = Chem.MolToSmiles(mol)
        for s in steps:
            stats[s.split(" of ")[0]] += 1
        out.append(Standardised(rec[id_key], smi, after, Chem.MolToInchiKey(mol),
                                after != Chem.MolToSmiles(Chem.MolFromSmiles(smi)),
                                "; ".join(steps)))
    stats["changed"] = sum(1 for s in out if s.changed)
    stats["total"] = len(out)
    return out, dict(stats)


# ---------------------------------------------------------------------------------------
# Property rules: flagged, never removed
# ---------------------------------------------------------------------------------------

def property_rule_flags(smiles: str) -> dict:
    """Lipinski Ro5, Astex Ro3 and the Pfizer 3/75 rule. A violation is a flag, not a verdict.

    Removing a compound for failing a developability heuristic would bias the very
    structure-activity relationship being fitted, so these travel with the row.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    logp = Crippen.MolLogP(mol)
    mw = Descriptors.MolWt(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
    ro5 = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    ro3 = sum([mw > 300, logp > 3, hbd > 3, hba > 3, rot > 3])
    return {"ro5_violations": ro5, "ro5_pass": ro5 <= 1,
            "ro3_violations": ro3, "ro3_pass": ro3 == 0,
            "pfizer_3_75_risk": bool(logp > 3 and tpsa < 75),
            "cLogP": logp, "MolWt": mw, "TPSA": tpsa, "HBD": hbd, "HBA": hba, "RotBonds": rot}


# ---------------------------------------------------------------------------------------
# Matched molecular pairs and activity cliffs
# ---------------------------------------------------------------------------------------

def _fp_gen(radius: int = 2, bits: int = 2048):
    return rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=bits)


def fingerprints(smiles: list[str], radius: int = 2, bits: int = 2048) -> list:
    gen = _fp_gen(radius, bits)
    out = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        out.append(None if mol is None else gen.GetFingerprint(mol))
    return out


@dataclass
class Cliff:
    a: str
    b: str
    similarity: float
    delta: float
    sali: float
    position: str = ""


def activity_cliffs(ids: list[str], smiles: list[str], y: list[float],
                    similarity_min: float = 0.8, delta_min: float = 1.0
                    ) -> tuple[list[Cliff], dict]:
    """Near-neighbour pairs with a large potency gap, scored by SALI.

    SALI = |delta y| / (1 - Tanimoto). A pair at similarity 1.0 would divide by zero, so the
    denominator is floored at a thousandth of a similarity unit and the pair is reported as
    the near-identical structure it is. The returned statistics say how many pairs were
    examined and how many were near neighbours at all, so "no cliffs" can be told apart
    from "no near neighbours to have a cliff between".
    """
    fps = fingerprints(smiles)
    out: list[Cliff] = []
    pairs = neighbours = 0
    best_sim = 0.0
    for i in range(len(ids)):
        if fps[i] is None:
            continue
        for j in range(i + 1, len(ids)):
            if fps[j] is None:
                continue
            pairs += 1
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            best_sim = max(best_sim, sim)
            if sim < similarity_min:
                continue
            neighbours += 1
            d = abs(float(y[i]) - float(y[j]))
            if d < delta_min:
                continue
            out.append(Cliff(ids[i], ids[j], sim, d, d / max(1.0 - sim, 1e-3)))
    out.sort(key=lambda c: -c.sali)
    stats = {"pairs_examined": pairs, "near_neighbour_pairs": neighbours,
             "cliffs": len(out), "similarity_min": similarity_min,
             "delta_min": delta_min, "highest_similarity_seen": best_sim}
    return out, stats


def series_cliffs(series, y_by_id: dict[str, float], delta_min: float = 1.0) -> list[Cliff]:
    """Cliffs inside one congeneric series: members differing at exactly one position.

    This is the matched molecular pair in its strict sense - one structural change, one
    potency change - which is what makes the comparison interpretable at all.
    """
    positions = series.varying_positions() or series.positions
    out: list[Cliff] = []
    members = [m for m in series.members if m in y_by_id]
    for i, a in enumerate(members):
        for b in members[i + 1:]:
            diff = [p for p in positions if series.r_groups[a][p] != series.r_groups[b][p]]
            if len(diff) != 1:
                continue
            d = abs(y_by_id[a] - y_by_id[b])
            if d < delta_min:
                continue
            p = diff[0]
            out.append(Cliff(a, b, float("nan"), d, float("nan"),
                             f"{p}: {series.r_groups[a][p]} -> {series.r_groups[b][p]}"))
    out.sort(key=lambda c: -c.delta)
    return out


# ---------------------------------------------------------------------------------------
# Feature selection: the board's genetic algorithm, minimising intercorrelation
# ---------------------------------------------------------------------------------------

@dataclass
class GAResult:
    columns: tuple[int, ...]
    names: list[str]
    fitness: float
    q2: float
    mean_abs_r: float
    history: list[float] = field(default_factory=list)
    generations: int = 0


def mean_abs_intercorrelation(X: np.ndarray, cols) -> float:
    cols = list(cols)
    if len(cols) < 2:
        return 0.0
    sub = X[:, cols]
    keep = [k for k in range(sub.shape[1]) if sub[:, k].std() > 0]
    if len(keep) < 2:
        return 0.0
    c = np.corrcoef(sub[:, keep], rowvar=False)
    iu = np.triu_indices_from(c, k=1)
    return float(np.nanmean(np.abs(c[iu])))


def genetic_feature_selection(X: np.ndarray, y: np.ndarray, names: list[str],
                              fitness_fn, max_features: int,
                              groups: list[tuple[int, ...]] | None = None,
                              population: int = 60, generations: int = 40,
                              correlation_penalty: float = 0.35,
                              mutation: float = 0.12, elite: int = 6,
                              seed: int = 0) -> GAResult:
    """Binary-mask GA over selection units, scoring quality minus mean |r| among the picks.

    `fitness_fn(X_sub, y) -> float` is supplied by the caller so the GA never decides what
    "good" means; the penalty term is the board's instruction to minimise intercorrelation.
    Units come from the same grouping rule as subset selection, so a parabolic pair is one
    gene and can never be half-selected.
    """
    from qsar_core import _units  # same unit rule as best_subset_aicc

    units = _units(X.shape[1], groups)
    rng = np.random.default_rng(seed)
    n_units = len(units)
    cap = min(max_features, n_units)

    def columns_of(mask) -> tuple[int, ...]:
        return tuple(j for k, u in enumerate(units) if mask[k] for j in u)

    def score(mask) -> tuple[float, float, float]:
        cols = columns_of(mask)
        if not cols:
            return -1e9, float("nan"), float("nan")
        q = fitness_fn(X[:, list(cols)], y)
        r = mean_abs_intercorrelation(X, cols)
        return q - correlation_penalty * r, q, r

    def random_mask():
        k = int(rng.integers(1, cap + 1))
        m = np.zeros(n_units, dtype=bool)
        m[rng.permutation(n_units)[:k]] = True
        return m

    def repair(m):
        on = np.flatnonzero(m)
        if len(on) == 0:
            m[rng.integers(n_units)] = True
        elif len(on) > cap:
            m[rng.permutation(on)[cap:]] = False
        return m

    pop = [random_mask() for _ in range(population)]
    scored = [(score(m)[0], m) for m in pop]
    history = []
    for _ in range(generations):
        scored.sort(key=lambda t: -t[0])
        history.append(scored[0][0])
        nxt = [m for _, m in scored[:elite]]
        while len(nxt) < population:
            a, b = (scored[int(rng.integers(0, max(2, population // 2)))][1] for _ in range(2))
            cut = int(rng.integers(1, n_units)) if n_units > 1 else 1
            child = np.concatenate([a[:cut], b[cut:]]).copy()
            flip = rng.random(n_units) < mutation
            child[flip] = ~child[flip]
            nxt.append(repair(child))
        scored = [(score(m)[0], m) for m in nxt]
    scored.sort(key=lambda t: -t[0])
    best = scored[0][1]
    fit, q, r = score(best)
    history.append(fit)
    cols = columns_of(best)
    return GAResult(cols, [names[i] for i in cols], fit, q, r, history, generations)


# ---------------------------------------------------------------------------------------
# Atom-level attribution and similarity maps
# ---------------------------------------------------------------------------------------

def atom_weights(mol, predict, radius: int = 2, bits: int = 2048) -> list[float]:
    """Per-atom contribution: the prediction minus the prediction with that atom masked out.

    This is what RDKit's own similarity-map helper does internally; doing it here means the
    weights can be inspected, reused and tested without depending on a drawing call.
    `predict` takes one fingerprint bit vector and returns a number.
    """
    import warnings
    from rdkit.Chem.Draw import SimilarityMaps

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        full = float(predict(SimilarityMaps.GetMorganFingerprint(
            mol, radius=radius, fpType="bv", nBits=bits)))
        out = []
        for i in range(mol.GetNumAtoms()):
            masked = SimilarityMaps.GetMorganFingerprint(
                mol, atomId=i, radius=radius, fpType="bv", nBits=bits)
            out.append(full - float(predict(masked)))
    return out


def similarity_map_png(mol, weights: list[float], size: tuple[int, int] = (460, 400)) -> bytes:
    """A similarity map as PNG bytes, across the RDKit versions that changed this signature.

    RDKit 2023 and later require an explicit drawer; older releases return a matplotlib
    figure. Both paths are taken here so the notebook renders on whatever Colab ships.
    """
    import warnings
    from rdkit.Chem.Draw import SimilarityMaps, rdMolDraw2D

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            d = rdMolDraw2D.MolDraw2DCairo(*size)
            SimilarityMaps.GetSimilarityMapFromWeights(mol, list(weights), draw2d=d)
            d.FinishDrawing()
            return d.GetDrawingText()
        except TypeError:
            import io
            fig = SimilarityMaps.GetSimilarityMapFromWeights(mol, list(weights), colorMap="bwr")
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
            return buf.getvalue()

In [ ]:
%%writefile tools/splits.py
"""One split object, built once and reused by every tier, plus the applicability domain.

Comparing four tiers on four different splits would compare the splits, not the tiers. So
the split is made once here - Bemis-Murcko scaffold clusters, with a Tanimoto cutoff that
keeps a test molecule from being a near-duplicate of a training one - and every tier is
handed the same object.
"""
from __future__ import annotations

import collections
from dataclasses import dataclass, field

import numpy as np
from rdkit import DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")


def scaffold_of(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def scaffold_clusters(ids: list[str], smiles: list[str]) -> dict[str, str]:
    """compound id -> scaffold SMILES. The scaffold is the grouping GroupKFold respects."""
    out = {}
    for cid, smi in zip(ids, smiles):
        sc = scaffold_of(smi)
        if sc is not None:
            out[cid] = sc
    return out


@dataclass
class Split:
    train: list[str]
    test: list[str]
    groups: dict[str, str]
    folds: list[tuple[list[str], list[str]]] = field(default_factory=list)
    near_duplicates_moved: list[tuple[str, str, float]] = field(default_factory=list)
    report: dict = field(default_factory=dict)


def _fps(smiles: list[str]):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    out = []
    for smi in smiles:
        m = Chem.MolFromSmiles(smi)
        out.append(None if m is None else gen.GetFingerprint(m))
    return out


def scaffold_split(ids: list[str], smiles: list[str], test_fraction: float = 0.25,
                   tanimoto_cutoff: float = 0.95, n_folds: int = 5,
                   seed: int = 0) -> Split:
    """Whole scaffolds to one side or the other, largest first, then a near-duplicate sweep.

    Assigning whole scaffolds means a test compound never shares its core with a training
    one. The sweep afterwards catches the remaining case the scaffold rule cannot see: two
    different cores whose whole molecules are still near-identical by fingerprint. Such a
    test compound is moved into training, and the move is recorded.
    """
    groups = scaffold_clusters(ids, smiles)
    by_scaffold: dict[str, list[str]] = collections.defaultdict(list)
    for cid in ids:
        if cid in groups:
            by_scaffold[groups[cid]].append(cid)
    order = sorted(by_scaffold, key=lambda s: (-len(by_scaffold[s]), s))
    target = int(round(test_fraction * sum(len(v) for v in by_scaffold.values())))
    test: list[str] = []
    # Largest scaffolds to training, so the test set is the sparse chemotypes: the harder
    # and more honest direction for a scaffold split.
    for sc in reversed(order):
        if len(test) >= target:
            break
        test.extend(by_scaffold[sc])
    test_set = set(test)
    train = [c for c in ids if c in groups and c not in test_set]

    smi_of = dict(zip(ids, smiles))
    fp_train = _fps([smi_of[c] for c in train])
    moved = []
    for cid in list(test):
        fp = _fps([smi_of[cid]])[0]
        if fp is None:
            continue
        sims = [(DataStructs.TanimotoSimilarity(fp, f), train[k])
                for k, f in enumerate(fp_train) if f is not None]
        if not sims:
            continue
        s, nearest = max(sims)
        if s >= tanimoto_cutoff:
            moved.append((cid, nearest, float(s)))
            test.remove(cid)
            train.append(cid)
    rng = np.random.default_rng(seed)
    split = Split(train=train, test=test, groups=groups, near_duplicates_moved=moved)
    split.folds = group_k_fold(train, groups, n_folds=n_folds, seed=seed)
    split.report = {
        "n_total": len(ids), "n_train": len(train), "n_test": len(test),
        "scaffolds_total": len(by_scaffold),
        "scaffolds_train": len({groups[c] for c in train}),
        "scaffolds_test": len({groups[c] for c in test}),
        "shared_scaffolds": len({groups[c] for c in train} & {groups[c] for c in test}),
        "near_duplicates_moved": len(moved),
        "tanimoto_cutoff": tanimoto_cutoff,
        "folds": len(split.folds),
        "fold_sizes": [len(te) for _, te in split.folds],
    }
    return split


def group_k_fold(ids: list[str], groups: dict[str, str], n_folds: int = 5,
                 seed: int = 0) -> list[tuple[list[str], list[str]]]:
    """Folds over scaffold clusters: a scaffold is never split across a fold boundary.

    Written out rather than imported so the assignment rule is visible: scaffolds are taken
    largest first and each goes to whichever fold is currently smallest, which balances fold
    sizes without ever cutting a cluster in two.
    """
    members: dict[str, list[str]] = collections.defaultdict(list)
    for cid in ids:
        if cid in groups:
            members[groups[cid]].append(cid)
    buckets: list[list[str]] = [[] for _ in range(n_folds)]
    for sc in sorted(members, key=lambda s: (-len(members[s]), s)):
        smallest = min(range(n_folds), key=lambda k: (len(buckets[k]), k))
        buckets[smallest].extend(members[sc])
    folds = []
    for k in range(n_folds):
        test = buckets[k]
        train = [c for j in range(n_folds) if j != k for c in buckets[j]]
        if test and train:
            folds.append((train, test))
    return folds


# ---------------------------------------------------------------------------------------
# Applicability domain
# ---------------------------------------------------------------------------------------

@dataclass
class Domain:
    h_star: float
    leverage: np.ndarray
    inside_leverage: np.ndarray
    inside_box: np.ndarray
    inside_both: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    report: dict = field(default_factory=dict)


def applicability_domain(X_train: np.ndarray, X_query: np.ndarray,
                         percentile: float = 1.0) -> Domain:
    """Hat leverage against 3p/n, and a percentile bounding box over the training ranges.

    Two criteria because they fail differently: leverage catches a compound far from the
    training centroid in the fitted space, the box catches one outside the observed range of
    any single descriptor. A prediction outside either is labelled, not quietly reported.
    """
    from qsar_core import leverage_of

    Xtr = np.asarray(X_train, dtype=float)
    Xq = np.asarray(X_query, dtype=float)
    if Xtr.ndim == 1:
        Xtr, Xq = Xtr.reshape(-1, 1), Xq.reshape(-1, 1)
    p = Xtr.shape[1]
    h_star = 3.0 * (p + 1) / len(Xtr)
    lev = leverage_of(Xtr, Xq)
    lower = np.percentile(Xtr, percentile, axis=0)
    upper = np.percentile(Xtr, 100.0 - percentile, axis=0)
    in_box = np.all((Xq >= lower) & (Xq <= upper), axis=1)
    in_lev = lev <= h_star
    dom = Domain(h_star=h_star, leverage=lev, inside_leverage=in_lev, inside_box=in_box,
                 inside_both=in_lev & in_box, lower=lower, upper=upper)
    dom.report = {"h_star": h_star, "percentile": percentile, "n_query": len(Xq),
                  "inside_leverage": int(in_lev.sum()), "inside_box": int(in_box.sum()),
                  "inside_both": int((in_lev & in_box).sum()),
                  "outside_either": int((~(in_lev & in_box)).sum())}
    return dom

In [ ]:
%%writefile tools/schematics.py
"""The two figures the board asks for, generated from the data rather than drawn.

One layout model, two renderers: an `.excalidraw` scene that opens in the same editor the
board came from, and a matplotlib figure so the notebook can show it inline without any
external renderer. Both read the same `Diagram`, so the picture in the notebook and the
file handed back cannot drift apart.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field

PALETTE = {
    "experimental": {"bg": "#e7f5e9", "stroke": "#2f7a3f"},
    "computed":     {"bg": "#eaf0fb", "stroke": "#2f5d9e"},
    "annotation":   {"bg": "#f6f1e7", "stroke": "#8a6d2f"},
    "derived":      {"bg": "#f7e9f2", "stroke": "#8e3a6b"},
    "step":         {"bg": "#f2f3f5", "stroke": "#4a4f57"},
}


@dataclass
class Box:
    key: str
    label: str
    kind: str
    x: float
    y: float
    w: float = 280.0
    h: float = 68.0
    note: str = ""


@dataclass
class Edge:
    a: str
    b: str
    label: str = ""


@dataclass
class Diagram:
    title: str
    boxes: list[Box] = field(default_factory=list)
    edges: list[Edge] = field(default_factory=list)

    def box(self, key: str) -> Box:
        return next(b for b in self.boxes if b.key == key)


def schema_diagram(columns: list[tuple[str, str, str]], title: str) -> Diagram:
    """columns: (name, kind, note). Laid out in kind-blocks, as the board draws the schema."""
    order = ["annotation", "experimental", "computed", "derived"]
    d = Diagram(title=title)
    groups: dict[str, list[tuple[str, str, str]]] = {k: [] for k in order}
    for name, kind, note in columns:
        groups.setdefault(kind, []).append((name, kind, note))
    x = 40.0
    for kind in order:
        rows = groups.get(kind) or []
        if not rows:
            continue
        y = 110.0
        for name, _, note in rows:
            d.boxes.append(Box(f"{kind}:{name}", name, kind, x, y, 260, 56, note))
            y += 70
        d.boxes.append(Box(f"hdr:{kind}", kind.upper(), kind, x, 30.0, 260, 48))
        x += 300
    return d


def workflow_diagram(steps: list[tuple[str, str, str]], title: str) -> Diagram:
    """steps: (section, label, board stage). One column, top to bottom, arrows between."""
    d = Diagram(title=title)
    y = 30.0
    for section, label, stage in steps:
        d.boxes.append(Box(section, f"{section}  {label}", "step", 40.0, y, 460, 60,
                           f"board stage {stage}"))
        y += 86
    for a, b in zip(d.boxes, d.boxes[1:]):
        d.edges.append(Edge(a.key, b.key))
    return d


# ---------------------------------------------------------------------------------------
# Renderers
# ---------------------------------------------------------------------------------------

def _el(i: int, **kw) -> dict:
    base = {"id": f"el{i}", "angle": 0, "strokeWidth": 1.5, "strokeStyle": "solid",
            "fillStyle": "solid", "roughness": 1, "opacity": 100, "groupIds": [],
            "frameId": None, "roundness": {"type": 3}, "seed": 1000 + i, "version": 1,
            "versionNonce": 1, "isDeleted": False, "boundElements": None,
            "updated": 1, "link": None, "locked": False}
    base.update(kw)
    return base


def to_excalidraw(d: Diagram) -> dict:
    els: list[dict] = []
    i = 0
    ids: dict[str, str] = {}
    for b in d.boxes:
        c = PALETTE.get(b.kind, PALETTE["step"])
        i += 1
        ids[b.key] = f"el{i}"
        els.append(_el(i, type="rectangle", x=b.x, y=b.y, width=b.w, height=b.h,
                       strokeColor=c["stroke"], backgroundColor=c["bg"]))
        i += 1
        text = b.label + (f"\n{b.note}" if b.note else "")
        els.append(_el(i, type="text", x=b.x + 12, y=b.y + 12, width=b.w - 24,
                       height=b.h - 24, strokeColor=c["stroke"],
                       backgroundColor="transparent", text=text, fontSize=16,
                       fontFamily=1, textAlign="left", verticalAlign="top",
                       originalText=text, lineHeight=1.25, roundness=None))
    for e in d.edges:
        a, b = d.box(e.a), d.box(e.b)
        i += 1
        els.append(_el(i, type="arrow", x=a.x + a.w / 2, y=a.y + a.h,
                       width=(b.x + b.w / 2) - (a.x + a.w / 2), height=b.y - (a.y + a.h),
                       strokeColor="#4a4f57", backgroundColor="transparent",
                       points=[[0, 0], [(b.x + b.w / 2) - (a.x + a.w / 2), b.y - (a.y + a.h)]],
                       startBinding={"elementId": ids[e.a], "focus": 0, "gap": 4},
                       endBinding={"elementId": ids[e.b], "focus": 0, "gap": 4},
                       startArrowhead=None, endArrowhead="arrow", roundness={"type": 2}))
    return {"type": "excalidraw", "version": 2,
            "source": "drug-hansch-space/notebook/tools/schematics.py",
            "elements": els, "appState": {"viewBackgroundColor": "#ffffff",
                                          "gridSize": None}, "files": {}}


def write_excalidraw(d: Diagram, path) -> str:
    text = json.dumps(to_excalidraw(d), indent=1)
    with open(path, "w") as fh:
        fh.write(text)
    return text


def render(d: Diagram, ax=None, scale: float = 1 / 90):
    """Draw the same Diagram with matplotlib, for display where no editor is available."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

    xs = [b.x + b.w for b in d.boxes] or [600]
    ys = [b.y + b.h for b in d.boxes] or [400]
    if ax is None:
        _, ax = plt.subplots(figsize=(max(xs) * scale, max(ys) * scale))
    for b in d.boxes:
        c = PALETTE.get(b.kind, PALETTE["step"])
        ax.add_patch(FancyBboxPatch((b.x, -b.y - b.h), b.w, b.h,
                                    boxstyle="round,pad=2,rounding_size=8",
                                    linewidth=1.4, edgecolor=c["stroke"],
                                    facecolor=c["bg"]))
        ax.text(b.x + 12, -b.y - 22, b.label, fontsize=9, color=c["stroke"],
                va="center", ha="left", fontweight="bold")
        if b.note:
            ax.text(b.x + 12, -b.y - 42, b.note, fontsize=7.5, color=c["stroke"],
                    va="center", ha="left", alpha=0.85)
    for e in d.edges:
        a, b = d.box(e.a), d.box(e.b)
        ax.add_patch(FancyArrowPatch((a.x + a.w / 2, -a.y - a.h),
                                     (b.x + b.w / 2, -b.y),
                                     arrowstyle="-|>", mutation_scale=11,
                                     linewidth=1.1, color="#4a4f57"))
    ax.set_xlim(0, max(xs) + 40)
    ax.set_ylim(-max(ys) - 40, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(d.title, fontsize=11, loc="left")
    return ax

In [ ]:
%%writefile tools/whiteboard_sections.py
"""The reading of the whiteboard: which cards belong to which stage of the workflow.

`parse_whiteboard.py` recovers the board's graph mechanically. This file is the one
interpretive step, and it is deliberately explicit so it can be argued with: each stage
lists the card numbers that `whiteboard_cards.tsv` gives them, in that file's order, and
every labelled card on the board appears in exactly one stage — including the stages that
this notebook does not implement, which are named `out_of_scope` with the reason.

Card numbers come from parse_whiteboard.py's reading order (top-to-bottom in 220px bands,
left-to-right inside a band) and are stable for a given scene file.
"""

SCENE_SHA256_PREFIX = None  # filled in by the mapping generator

STAGES = [
    {
        "key": "A_sources",
        "title": "A. Where the data comes from",
        "cards": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 23, 24, 25, 26, 27, 28, 32, 33, 34, 35,
                  36, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 250, 251],
        "reading": (
            "A decision tree over sources. It starts at PyTDC / Therapeutic Data Commons and "
            "walks a series of questions - group by target receptor, phenotypic side effects, "
            "clinical trials, standardised activity values, pharmacokinetic exposure, endpoints "
            "not in TDC - to Tox21, ToxCast, ClinTox, SIDER, ChEMBL, and admetSAR 3.0 / "
            "QSAR Toolbox. A separate table lists PubChem, ChEMBL, DrugBank, Papyrus, the "
            "Harmonized Drug Dataset Quality Control set, and DailyMed."
        ),
        "implemented_by": "nb:sources",
    },
    {
        "key": "B_schema",
        "title": "B. The dataset's schema",
        "cards": [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 29, 30, 31, 40,
                  54, 55, 56, 57, 58, 59, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78,
                  79, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 91, 92, 93, 96],
        "reading": (
            "One row per drug, carrying an identifier block (PubChem ID, generic name, "
            "structural identifier, National Drug Code), an annotation block (FDA label, "
            "DrugBank, ATC code) and the two measured blocks, Activity and Tox. The worked "
            "example is the ATC code split into its four levels: cimetidine is one code "
            "(A / 02 / BA / 01) and mannitol is four, which is the '1 Drug 1 ATC Code?' "
            "decision - a drug with several codes becomes several rows."
        ),
        "implemented_by": "nb:schema",
    },
    {
        "key": "C_preprocess",
        "title": "C. Pre-processing and outlier removal",
        "cards": [49, 50, 51, 52, 53, 60, 61, 62, 63, 64, 65, 66, 67, 88, 125],
        "reading": (
            "Remove duplicates, drop rows with missing values, standardise. Then two outlier "
            "tests with different grounds: a property-rule violation (Rule of 5, Rule of 3, "
            "Pfizer's rule) marks a property outlier, and a matched molecular pair inside a "
            "congeneric series with a large pIC50 difference marks a structural outlier - an "
            "activity cliff. Both are removed, and the board asks for the justification to be "
            "written down."
        ),
        "implemented_by": "nb:preprocess",
    },
    {
        "key": "D_selection",
        "title": "D. Selecting the modelling set, and the empty arrow",
        "cards": [104, 105, 106, 108, 112, 247],
        "reading": (
            "Three selection criteria: a common target, standardised units, orthogonal assay "
            "data. The target variable is the Therapeutic Index. One box - 'What are the "
            "properties/structural features?' - has an arrow leading out of it and nothing on "
            "the end: this is the box left for me to fill. A second open question asks what "
            "happens if toxicity is used as a predictor of activity."
        ),
        "implemented_by": "nb:descriptors",
    },
    {
        "key": "E_features",
        "title": "E. Feature selection and dimensionality reduction",
        "cards": [109, 110, 111, 113, 114, 119, 120, 149],
        "reading": (
            "Feature selection by a genetic algorithm whose objective minimises pairwise "
            "intercorrelation. Dimensionality reduction is a yes/no branch over PCA, LDA and "
            "t-SNE (deeptime, openTSNE). Descriptors come from the MORDRED library."
        ),
        "implemented_by": "nb:descriptors,nb:tier1",
    },
    {
        "key": "F_models",
        "title": "F. Model building",
        "cards": [95, 97, 98, 100, 101, 102, 103, 94],
        "reading": (
            "Two columns, regression for a continuous target and classification for a discrete "
            "one, each with its dependencies named. The regression column is led by 'Hansch "
            "Analysis based Multi Linear Regression' and continues through principal component "
            "regression, PLS and kernel PLS, SVR with a Gaussian kernel, random forest, "
            "gradient boosting, MLP and a graph neural network. Libraries: QSPRpred, "
            "QSAR-Co-X, PyQSAR3."
        ),
        "implemented_by": "nb:tier1,nb:tier2,nb:tier3,nb:tier4",
    },
    {
        "key": "G_tuning",
        "title": "G. Hyperparameter tuning",
        "cards": [107, 115, 121, 122],
        "reading": "Optuna, with median pruning and successive halving.",
        "implemented_by": "nb:tier4",
    },
    {
        "key": "H_validation",
        "title": "H. Validation",
        "cards": [116, 117, 118, 123, 124, 150, 153, 154, 155, 156, 157, 161, 162,
                  164, 165, 166, 167, 129, 133, 134, 135],
        "reading": (
            "Internal validation by Y-randomisation, k-fold, nested cross-validation, "
            "leave-one-out and leave-many-out, reported as R-squared, Q-squared and RMSE. "
            "External validation by the concordance correlation coefficient, citing Chirico "
            "and Gramatica 2011. The split is a scaffold split: Morgan fingerprints, "
            "Bemis-Murcko scaffolds and a Tanimoto cutoff so test molecules are not too "
            "similar to training ones, with Group K-Fold over the clusters; eight clustering "
            "methods are listed, and each cluster is to be checked against a co-crystallised "
            "ligand from the PDB."
        ),
        "implemented_by": "nb:validation,nb:cv",
    },
    {
        "key": "I_interpretation",
        "title": "I. Interpretation",
        "cards": [126, 127, 128, 130, 131, 136, 137],
        "reading": (
            "SHAP, Morgan fingerprint similarity maps, a structure-activity landscape index, "
            "and Hansch-based mechanistic interpretation."
        ),
        "implemented_by": "nb:tier1,nb:interpret",
    },
    {
        "key": "J_domain",
        "title": "J. Applicability domain",
        "cards": [146, 147, 148],
        "reading": (
            "Five metrics: hat leverage, autoencoder error in SMILES reconstruction, a "
            "percentile bounding box over the feature distribution, an envelope method and a "
            "distance method."
        ),
        "implemented_by": "nb:domain",
    },
    {
        "key": "K_hansch",
        "title": "K. The Hansch core",
        "cards": [138, 139, 140, 141, 142, 143, 144, 145, 151, 152, 158, 159, 160, 163,
                  217, 218, 219, 224, 225, 226, 227, 229, 230, 231, 232, 233, 235, 236,
                  237, 240, 241, 242, 243, 244, 132],
        "reading": (
            "The theory, drawn out. Dose-response is sigmoidal with a linear region; AC50 is "
            "read off it and the response variable is log(1/C), or log of the reciprocal of the "
            "minimum effective concentration when the response is binary. Binding energy is the "
            "natural log of the association constant, so a lower AC50 means tighter binding and "
            "a more negative binding-energy change. The linear free-energy assumption: the total "
            "change in binding energy is the sum of the hydrophobic, electronic and steric "
            "contributions of a substituent, with the parabola in log P because a drug must be "
            "water-soluble enough to travel and lipid-soluble enough to enter cells. Hammett's "
            "constant is positive for electron-withdrawing groups; the Taft steric factor is "
            "negative for bulky ones. The equation is written out as "
            "log(1/C) = -a(logP)^2 + b logP + c*sigma + d*Es + error. Two caveats are written on "
            "the board: the relationship needs enough aqueous solubility, and molecules in one "
            "congeneric series can still show activity cliffs. A third says Hansch results should "
            "be checked against binding-energy changes from docking."
        ),
        "implemented_by": "nb:tier1,nb:tier2",
    },
    {
        "key": "L_goal",
        "title": "L. The goal",
        "cards": [246, 248, 99],
        "reading": (
            "'Build and Integrate Multiple Local QSAR models into a single Global QSAR model for "
            "prediction within a therapeutically diverse dataset (Different Anatomical "
            "Therapeutic Chemical Code Levels) curated by Paccanaro Research Labs', written "
            "twice, with the Galeano and Paccanaro 2022 Cell Reports Methods paper cited."
        ),
        "implemented_by": "nb:goal",
    },
    {
        "key": "out_of_scope_clinical",
        "title": "Out of scope here: clinical trials, pharmacokinetics and epidemiology",
        "cards": [168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,
                  182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195,
                  196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209,
                  210, 211, 212, 213, 214, 215, 216, 220, 221, 222, 223, 228, 234, 238,
                  239, 245, 249],
        "reading": (
            "A large block of study notes alongside the pipeline: clearance and AUC, relative "
            "bioavailability, trial design (randomisation, placebo, double blind, open label, "
            "phase 1), reading relative risk claims honestly, four models for paying trial "
            "participants, and prevalence against incidence."
        ),
        "implemented_by": None,
        "reason_out_of_scope": (
            "These cards are about clinical trial conduct and epidemiology, not about fitting or "
            "validating a QSAR model. Nothing in them feeds a descriptor, an endpoint or a "
            "validation step, so the notebook does not implement them. They are listed here so "
            "it is clear they were read and not missed."
        ),
    },
]

CELL_IDS = {
    "nb:sources": "Section 3, cell `source_pull`",
    "nb:schema": "Section 4, cells `schema` and `atc_split`",
    "nb:preprocess": "Section 5, cells `dedupe_standardise`, `rule_flags`, `cliffs`",
    "nb:descriptors": "Section 6, cells `modelling_set`, `select_series`, `descriptor_block`",
    "nb:tier1": "Section 7, cells `tier1_hansch`, `tier1_subsets`, `tier1_diagnostics`",
    "nb:tier2": "Section 8, cell `tier2_free_wilson`",
    "nb:tier3": "Section 9, cells `tier3_ga`, `tier3_pls_pca`",
    "nb:tier4": "Section 10, cells `tier4_optuna`, `tier4_fit`",
    "nb:cv": "Section 11, cell `group_k_fold`",
    "nb:validation": "Section 11, cells `internal_validation`, `external_validation`",
    "nb:domain": "Section 12, cell `applicability_domain`",
    "nb:interpret": "Section 13, cells `tier1_reading`, `shap_values`, `similarity_maps`",
    "nb:goal": "Section 2, markdown `goal`, and Section 15, cell `comparison_table`",
}

# Model families the board's regression column names, and whether this notebook fits them.
# Listed so the gap is visible rather than implied away.
BOARD_MODELS = {
    "Hansch analysis based multi linear regression": "Tier 1, section 7",
    "Free-Wilson": "Tier 2, section 8",
    "Principal component regression": "Tier 3, section 9",
    "Partial least squares": "Tier 3, section 9",
    "Kernel PLS": None,
    "Support vector regression, Gaussian kernel": None,
    "Random forest": "Tier 4, section 10",
    "Gradient boosting": "Tier 4, section 10",
    "Multilayer perceptron": None,
    "Graph neural network": None,
}

In [ ]:
# Section 1b (check). Re-hash what was just written and compare with section 0's record.
drift = []
for name, expected in MODULE_SHA256.items():
    got = hashlib.sha256(open(f"tools/{name}.py", "rb").read()).hexdigest()
    if got != expected:
        drift.append((name, expected, got))
if drift:
    for name, exp, got in drift:
        print(f"  tools/{name}.py  expected {exp}  got {got}")
    raise SystemExit("the embedded module source does not match the hash recorded in section 0")
print(f"all {len(MODULE_SHA256)} modules written and hash-matched")

import curate, features, qsar_core, schematics, series, splits, substituent_constants
import whiteboard_sections
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
np.random.seed(SEED)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
RESULTS = {"environment": None, "data": {}, "models": {}, "validation": {}, "figures": []}
print("imports ready")

In [ ]:
# Section 1c. Resolved versions, after the install step. A mismatch with the declared pin is
# a fact about this run, so it is printed rather than corrected.
import importlib

print(f"{'library':<16}{'declared':<14}{'resolved':<14}match")
mismatch = []
for package, pin in DECLARED_PINS.items():
    mod = importlib.import_module(IMPORT_NAMES.get(package, package))
    got = getattr(mod, "__version__", "?")
    same = (got == pin)
    if not same:
        mismatch.append((package, pin, got))
    print(f"{package:<16}{pin:<14}{got:<14}{'yes' if same else 'NO'}")

RUN_ENVIRONMENT = {
    "python": platform.python_version(),
    "versions": {p: getattr(importlib.import_module(IMPORT_NAMES.get(p, p)), "__version__", "?")
                 for p in DECLARED_PINS},
    "pin_mismatches": [{"library": a, "declared": b, "resolved": c} for a, b, c in mismatch],
    "seed": SEED,
    "git_commit": commit,
}
if mismatch:
    print("\nThis run used", len(mismatch), "library version(s) other than the declared pin.")
    print("That is recorded in RUN_ENVIRONMENT and exported in section 16.")

## §2 The goal, and the board

The board's Goal 1 card, verbatim:

> **Build and Integrate Multiple Local QSAR models into a single Global QSAR model for prediction within a therapeutically diverse dataset (Different Anatomical Therapeutic Chemical Code Levels) curated by Paccanaro Research Labs.**

This notebook builds **one** such local model properly, on one target pair, with every step the board draws. One local model done to the standard the board sets is what a global assembly would have to be made of; assembling many of them is the next piece of work, not this one.

The stage-to-section table below is generated from the parsed board, not typed.

In [ ]:
# Section 2. The board's stages and where each one is implemented.
rows = []
for stage in whiteboard_sections.STAGES:
    keys = (stage["implemented_by"] or "").split(",") if stage["implemented_by"] else []
    where = "; ".join(whiteboard_sections.CELL_IDS[k] for k in keys if k) or "not implemented"
    rows.append({"stage": stage["title"], "cards": len(stage["cards"]), "notebook": where})
stage_table = pd.DataFrame(rows)
print(stage_table.to_string(index=False, max_colwidth=78))

print("\nModel families the board's regression column names:")
for model, where in whiteboard_sections.BOARD_MODELS.items():
    print(f"  {model:<46}{where or 'NOT FITTED HERE'}")
fitted = sum(1 for v in whiteboard_sections.BOARD_MODELS.values() if v)
print(f"\n{fitted} of {len(whiteboard_sections.BOARD_MODELS)} named model families are fitted below.")

out_of_scope = [s for s in whiteboard_sections.STAGES if not s["implemented_by"]]
for s in out_of_scope:
    print(f"\nRead and deliberately excluded: {s['title']} ({len(s['cards'])} cards)")
    print(textwrap.fill(s["reason_out_of_scope"], 96, initial_indent="  ", subsequent_indent="  "))

RESULTS["data"]["board_stages"] = rows
RESULTS["data"]["board_models"] = dict(whiteboard_sections.BOARD_MODELS)

## §3 Data ingestion — ChEMBL, with provenance on every row

### CHECKPOINT 1 — the data, and where each number came from

A live pull from the ChEMBL REST API. No key, no download, no cached copy: if this cell runs, the data is current, and the release it ran against is printed next to the row hash so the hash pins something.

Documents are chosen because each measures **both** endpoints on the **same compounds** in the **same paper**. A therapeutic index formed across two unrelated assays has no referent, which is the whole reason for choosing this way. That requirement is severe: in `ChEMBL_37` exactly **four** documents satisfy it with ten compounds or more, and `curate.document_scan()` below re-runs the search that found them rather than asking for the list to be taken on trust.

| | document | H3 activity | hERG toxicity | paired |
|---|---|---|---|---|
| training | CHEMBL1821550, *BMCL* 2011 | Ki, [125I]iodoproxyfan | IC50, [3H]dofetilide | 224 |
| external | CHEMBL1155513, *BMCL* 2010 | Ki, [3H]N-α-methylhistamine | Ki, [3H]dofetilide, HEK293 | 75 |
| external | CHEMBL2029265 | Ki, [3H]-(R)-α-methylhistamine | IC50, **IonWorks patch clamp** | 20 |
| external | CHEMBL2375287 | Ki, [3H]histamine | Ki, [3H]astemizole | 13 |

Three external documents, not one, and that is a **declared change to the plan**. The plan named CHEMBL1155513 as the external set on the strength of its 75 paired compounds. It turns out that **67 of those 75 compounds are also in the training paper** — the 2011 paper re-measures the 2010 series — leaving 8 unseen compounds, which is not an external set. Sections 5 and 11 print that overlap, and the other two documents supply 20 and 13 further compounds the training paper never saw. Their readouts differ from the training paper's and from each other, including one hERG number that is a **functional electrophysiology** measurement rather than a displacement binding affinity, so metrics are reported **per document as well as pooled**: a spread between laboratories is information, and averaging it away would hide the thing worth knowing.

Each filter prints rows in, rows out and the reason it exists. One filter from the original plan was also **changed after checking what it actually removes** — see the note after the ledger.

In [ ]:
# Section 3. The pull, filter by filter.
pull = curate.build()
release = pull["chembl"]
print(f"ChEMBL {release['release']}, released {release['release_date']}, "
      f"{release['activities']:,} activities in the release")
print(f"pulled at {release['pulled_at']}")

for name, report in pull["reports"].items():
    print(f"\n{'='*96}\n{name}: document {report['document']}  "
          f"({report['rows_in_document']} activity rows in the document)")
    cite = pull["citations"][name]
    print(f"  {cite['journal']} {cite['year']}  doi:{cite['doi']}  pmid:{cite['pubmed_id']}")
    print(f"  {cite['title']}")
    ledger = pd.DataFrame(report["ledger"])
    print(ledger[["filter", "rows_in", "rows_out", "dropped", "reason"]]
          .to_string(index=False, max_colwidth=66))
    print("  target / standard_type pairs surviving:", report["surviving_pairs"])
    for role, ep in report["endpoints"].items():
        print(f"  {role:<9}{ep['target']} {ep['standard_type']:<5} n={ep['rows']:<5}"
              f"assay {ep['assay_chembl_id']}")
        print(f"            {ep['assay_description'][:88]}")
        print(f"            rows carrying ChEMBL's potential_duplicate flag: "
              f"{ep['rows_carrying_potential_duplicate']}; compounds with more than one row: "
              f"{ep['compounds_with_several_rows']}")
    print("  counts:", report["counts"])

RESULTS["data"]["chembl_release"] = release
RESULTS["data"]["citations"] = pull["citations"]
RESULTS["data"]["ingestion_reports"] = pull["reports"]

In [ ]:
# Section 3 (search). The choice of documents, re-derived rather than asserted. This pages
# every binding activity for H3 and for hERG in the current release and intersects the
# molecule sets per document, so the table in the markdown above is checkable. One minute.
scan = curate.document_scan(min_paired=10)
scan_df = pd.DataFrame(scan)
scan_df["role"] = [next((n for n, sp in curate.SETS.items() if sp["document"] == d), "not used")
                   for d in scan_df["document"]]
print(f"documents in {release['release']} measuring BOTH H3 and hERG affinity on at least "
      f"10 shared compounds: {len(scan_df)}\n")
print(scan_df.to_string(index=False))

used = {sp["document"] for sp in curate.SETS.values()}
found = set(scan_df["document"])
assert used <= found, f"a configured document no longer pairs both endpoints: {used - found}"
if found - used:
    print("\ndocuments found but NOT used, and why:", found - used)
print("""
Every document the search finds is used. There is no fifth paper being left out, which also
means the external set cannot be made larger without relaxing the requirement that both
endpoints come from the same publication - and relaxing it is what makes a therapeutic index
meaningless, so it is not relaxed.""")
RESULTS["data"]["document_scan"] = scan

#### A filter from the plan that did not survive contact with the data

The plan listed `potential_duplicate = 0` among the filters. Checking what it removes:

* in the training document it would drop **159 of 458** surviving rows, including **102 of 234** H3 measurements and **57 of 207** hERG measurements;
* every one of those rows is the **only** row for its compound at that target and standard type — the group-size histogram is `{1: n}` for all four endpoints;
* so the filter would not prevent a single duplication. It would delete measurements and cut the training set from 207 paired compounds to 70.

`potential_duplicate` is ChEMBL's note that the same measurement appears **elsewhere in the database**, typically because a later paper in the same series reports it again. The key that matters here is (compound, target, standard type) **inside one document**, and that is deduplicated explicitly instead: the flag is counted and reported, no row is dropped for carrying it, and the cell above prints how many compounds had more than one row — zero, in every case. Section 17 lists this as a declared change to the plan.

In [ ]:
# Section 3 (assertions). Two things must hold or the notebook stops.
records = pd.DataFrame(pull["records"])

bad_source = records.loc[records["source_type"] != "experimental"]
assert bad_source.empty, f"{len(bad_source)} records are not experimental measurements"

mixed = (records.groupby(["set_name", "endpoint"])["assay_chembl_id"].nunique())
assert (mixed == 1).all(), f"an endpoint mixes assays:\n{mixed[mixed > 1]}"

print("assertion 1 passed: every endpoint value is an experimental measurement, none predicted")
print("assertion 2 passed: one assay per endpoint per set\n")
print(records.groupby(["set_name", "endpoint", "standard_type", "assay_chembl_id"])
      .size().rename("rows").to_frame().to_string())

provenance = (records[["set_name", "endpoint", "target_chembl_id", "standard_type",
                       "assay_chembl_id", "citation", "source_type"]]
              .drop_duplicates().reset_index(drop=True))
print("\nProvenance table, in full:")
print(provenance.to_string(index=False))
RESULTS["data"]["provenance"] = provenance.to_dict("records")

## §4 Schema, and the ATC worked example

The board draws the dataset as four blocks: an identifier block, an annotation block, an Activity block and a Tox block. Every column is tagged `experimental`, `computed`, `annotation` or `derived`, and that tag travels with it to the end.

The board's own worked example — one drug with a single ATC code against one with several — runs here as code rather than as a claim, pulling the ATC classifications live and splitting the code into its five levels.

In [ ]:
# Section 4a. The schema, with every column's kind.
SCHEMA = [
    ("set",                     "annotation",   "which source document the row came from"),
    ("role",                    "annotation",   "training or external"),
    ("document",                "annotation",   "ChEMBL document id"),
    ("compound_id",             "annotation",   "ChEMBL molecule id"),
    ("smiles",                  "annotation",   "canonical SMILES from ChEMBL"),
    ("p_activity_H3",           "experimental", "-log10(Ki / M), target CHEMBL264"),
    ("activity_value_nM",       "experimental", "as reported, nM"),
    ("p_toxicity_hERG",         "experimental", "-log10(IC50 or Ki / M), target CHEMBL240"),
    ("toxicity_value_nM",       "experimental", "as reported, nM"),
    ("therapeutic_index",       "derived",      "p_activity - p_toxicity"),
    ("activity_assay",          "annotation",   "ChEMBL assay id"),
    ("toxicity_assay",          "annotation",   "ChEMBL assay id"),
    ("citation",                "annotation",   "DOI of the source paper"),
    ("sum_pi, sum_sigma, ...",  "experimental", "tabulated substituent constants, section 6"),
    ("cLogP, TPSA, Chi1v, ...", "computed",     "RDKit, section 6"),
]
schema_df = pd.DataFrame(SCHEMA, columns=["column", "kind", "meaning"])
print(schema_df.to_string(index=False))
print("\ncolumns by kind:", schema_df["kind"].value_counts().to_dict())
RESULTS["data"]["schema"] = SCHEMA

table_rows = curate.table(pull)
df = pd.DataFrame(table_rows)
print(f"\ncurated table: {len(df)} rows")
print(df.groupby(["role", "set"])[["p_activity_H3", "p_toxicity_hERG", "therapeutic_index"]]
      .agg(["count", "mean", "std", "min", "max"]).round(2).to_string())

In [ ]:
# Section 4b. The board's ATC example, as code: one ATC code against several.
def atc_of(name):
    hits = curate.paged("molecule", {"pref_name__iexact": name})
    if not hits:
        return name, []
    m = hits[0]
    return m["molecule_chembl_id"], list(m.get("atc_classifications") or [])

LEVELS = ["anatomical", "therapeutic", "pharmacological", "chemical", "substance"]

def split_atc(code):
    return dict(zip(LEVELS, [code[:1], code[:3], code[:4], code[:5], code]))

atc_rows = []
for drug in ("cimetidine", "mannitol"):
    cid, codes = atc_of(drug)
    print(f"{drug} ({cid}): {len(codes)} ATC code(s) -> {len(codes)} row(s)")
    for c in codes:
        atc_rows.append(dict(drug=drug, compound_id=cid, atc_code=c, **split_atc(c)))
atc_df = pd.DataFrame(atc_rows)
print()
print(atc_df.to_string(index=False) if len(atc_df) else "no ATC classifications returned")
print("\nOne drug with several ATC codes becomes several rows, and the code splits into its")
print("five levels as its own columns - which is what lets a global model be grouped by level.")
RESULTS["data"]["atc_example"] = atc_rows

## §5 Pre-processing

### CHECKPOINT 3 — the board's preprocessing chain, in its own order

Duplicate removal by InChIKey. Missing values **counted and reported, never imputed**. Structure standardisation with RDKit's `rdMolStandardize`: largest fragment, neutralise, canonical tautomer, with the before-and-after count printed.

Then the board's two outlier tests, kept apart because they mean different things:

* a **property-rule violation** (Lipinski Ro5, Astex Ro3, Pfizer 3/75) is **flagged, never removed** — dropping a compound for failing a developability heuristic would bias the very structure–activity relationship being fitted, so the flag travels with the row and section 7 reports the fit with and without;
* a **matched molecular pair with a large potency gap** inside a congeneric series is a genuine activity cliff. It is flagged, counted, and its effect on the fit reported.

In [ ]:
# Section 5a. Duplicates by InChIKey, missing values, standardisation.
paired = df[df["missing"].isna()].copy()
print(f"rows with both endpoints: {len(paired)} of {len(df)}")

missing_table = pd.DataFrame({
    "column": ["p_activity_H3", "p_toxicity_hERG", "smiles"],
    "missing": [df["p_activity_H3"].isna().sum(), df["p_toxicity_hERG"].isna().sum(),
                df["smiles"].isna().sum()],
})
missing_table["what happens"] = [
    "row kept in the curated table, excluded from any model that needs the activity",
    "row kept in the curated table, excluded from any model that needs the toxicity",
    "row cannot be used at all",
]
print("\nMissing values - counted, reported, and never imputed:")
print(missing_table.to_string(index=False, max_colwidth=70))

standardised, std_stats = features.standardise(paired.to_dict("records"))
print("\nStandardisation (largest fragment, neutralise, canonical tautomer):")
for k, v in sorted(std_stats.items()):
    print(f"  {k:<22}{v}")

paired["inchikey"] = [s.inchikey for s in standardised]
paired["smiles_standardised"] = [s.after for s in standardised]
paired["standardisation_note"] = [s.note for s in standardised]

dupes = paired.groupby(["set", "inchikey"]).size()
n_dupe = int((dupes > 1).sum())
print(f"\nInChIKey duplicates within a set: {n_dupe}")
before = len(paired)
paired = paired.drop_duplicates(subset=["set", "inchikey"], keep="first").reset_index(drop=True)
print(f"rows {before} -> {len(paired)} after InChIKey deduplication")

train_keys = set(paired.loc[paired["role"] == "training", "inchikey"])
print("\nCompounds already present in the TRAINING paper, per external document. A compound")
print("the model has seen cannot test prediction, however it is labelled, so these go.")
overlap_rows = []
for doc_set in [s for s in paired["set"].unique() if s != "training"]:
    keys = set(paired.loc[paired["set"] == doc_set, "inchikey"])
    seen = keys & train_keys
    overlap_rows.append({"external set": doc_set,
                         "document": paired.loc[paired["set"] == doc_set, "document"].iloc[0],
                         "paired compounds": len(keys), "already in training": len(seen),
                         "unseen": len(keys - seen)})
overlap_df = pd.DataFrame(overlap_rows)
print(overlap_df.to_string(index=False))
total_overlap = int(overlap_df["already in training"].sum())
paired = paired[~((paired["role"] == "external") & (paired["inchikey"].isin(train_keys)))]
paired = paired.reset_index(drop=True)
print(f"\n{total_overlap} external rows dropped as already seen; "
      f"{int((paired['role'] == 'external').sum())} unseen external compounds remain")
if overlap_df["already in training"].max() > 0.5 * overlap_df["paired compounds"].max():
    print("""
That first document is the plan's original external set, and most of it is the training set
wearing a different citation. Had it been used as published, the external metrics in section 11
would have been measured largely on compounds the model had already been fitted to - which is
how an external validation comes to look better than the model is. It is reported here rather
than quietly corrected.""")
RESULTS["data"]["standardisation"] = std_stats
RESULTS["data"]["inchikey_duplicates"] = n_dupe
RESULTS["data"]["train_external_overlap"] = overlap_rows

In [ ]:
# Section 5b. Property rules: flagged, never removed.
flags = pd.DataFrame([features.property_rule_flags(s) for s in paired["smiles_standardised"]])
paired = pd.concat([paired.reset_index(drop=True), flags.reset_index(drop=True)], axis=1)

summary = paired.groupby("set")[["ro5_pass", "ro3_pass", "pfizer_3_75_risk"]].mean().round(3)
print("fraction of compounds passing each rule (Pfizer column is fraction AT RISK):")
print(summary.to_string())
print("\nRo5 violation counts:")
print(paired.groupby(["set", "ro5_violations"]).size().rename("compounds").to_frame().to_string())
print(f"\nNo compound is removed for any of these. {int((~paired['ro5_pass']).sum())} compounds")
print("fail Lipinski and stay in; section 7 refits without them and reports both.")
RESULTS["data"]["rule_flags"] = {
    "ro5_fail": int((~paired["ro5_pass"]).sum()),
    "ro3_fail": int((~paired["ro3_pass"]).sum()),
    "pfizer_risk": int(paired["pfizer_3_75_risk"].sum()),
}

In [ ]:
# Section 5c. Activity cliffs: near neighbours with a large potency gap.
train_rows = paired[paired["set"] == "training"].reset_index(drop=True)
cliffs, cliff_stats = features.activity_cliffs(
    list(train_rows["compound_id"]), list(train_rows["smiles_standardised"]),
    list(train_rows["p_activity_H3"]), similarity_min=0.75, delta_min=1.0)
print("Fingerprint near-neighbour cliffs in the training set (Morgan r=2, 2048 bits):")
for k, v in cliff_stats.items():
    print(f"  {k:<26}{round(v, 4) if isinstance(v, float) else v}")
if cliffs:
    print("\n  top pairs by SALI = |delta p| / (1 - Tanimoto):")
    print(pd.DataFrame([{"a": c.a, "b": c.b, "tanimoto": round(c.similarity, 3),
                         "delta_p": round(c.delta, 2), "SALI": round(c.sali, 1)}
                        for c in cliffs[:10]]).to_string(index=False))
else:
    print("\n  no pair clears both thresholds; the count of near neighbours above tells you")
    print("  whether that is because the chemistry is diverse or because the potencies agree.")
RESULTS["data"]["cliffs"] = cliff_stats

## §6 The modelling set, the target variable, and the empty arrow

### CHECKPOINT 2 — what is being modelled, and with what

**Selection criteria**, from the board: a common target, standardised units, orthogonal assay data.

**Target variable.** `TI = pActivity − pToxicity`, both on the same −log10 molar scale, higher meaning a wider margin between binding the intended receptor and binding hERG *in vitro*. Both endpoints are also modelled on their own, because a difference hides which side moved.

**The empty arrow.** The board's box *"What are the properties/structural features?"* has an arrow leaving it and nothing on the end. Filled here with exactly the block the board's own equation card specifies, and nothing more:

* **π (or log P) as a single lipophilicity descriptor**, entered as the parabolic pair (π, π²). The pair enters and leaves the model together and counts as **one** descriptor in the Topliss–Costello ratio and in subset selection. The arithmetic degrees of freedom still spend two coefficients; both counts are printed side by side so they are never confused.
* **Hammett σ**, meta and para kept distinct.
* **Taft Es**.
* **Molar refractivity MR**.

Values come from the Hansch–Leo–Taft compilation and *Exploring QSAR* — tabulated experimental constants. Transcription is the weak point of any such table, so the next cell re-derives π from measured log P by its own definition, π(X) = log P(C₆H₅X) − log P(C₆H₆), using only measured log P values pulled from ChEMBL, and names any substituent that disagrees instead of using it silently.

RDKit-computed descriptors are allowed for Tiers 3 and 4 only, each tagged `computed`.

In [ ]:
# Section 6a. The modelling sets and the target variable, with the sign convention worked out.
model_df = paired[paired["role"] == "training"].reset_index(drop=True)
ext_df = paired[paired["role"] == "external"].reset_index(drop=True)
print(f"training modelling set: {len(model_df)} compounds")
print(f"external set          : {len(ext_df)} compounds, from "
      f"{ext_df['document'].nunique()} independent documents")
print(ext_df.groupby(["set", "document", "activity_standard_type", "toxicity_standard_type"])
      .size().rename("compounds").to_frame().to_string())

ENDPOINTS = {"p_activity_H3": "H3 affinity", "p_toxicity_hERG": "hERG affinity",
             "therapeutic_index": "pActivity - pToxicity"}

row = model_df.iloc[int(model_df["therapeutic_index"].idxmax())]
print(f"""
Sign convention, worked through on the widest-margin compound in the training set:
  {row['compound_id']}
  H3 Ki        {row['activity_value_nM']:>10.3g} nM  ->  pActivity = {row['p_activity_H3']:.2f}
  hERG {row['toxicity_standard_type']:<5}  {row['toxicity_value_nM']:>10.3g} nM  ->  pToxicity = {row['p_toxicity_hERG']:.2f}
  TI = pActivity - pToxicity = {row['therapeutic_index']:.2f}
  A larger TI means the compound binds H3 at a lower concentration than it binds hERG.
  It is a ratio of two in vitro binding affinities. It is not a dose, a margin in a patient,
  or a clinical therapeutic index.""")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
for ax, (col, label) in zip(axes, ENDPOINTS.items()):
    ax.hist(model_df[col], bins=24, color="#2f5d9e", alpha=0.85, label="training")
    ax.hist(ext_df[col], bins=24, color="#c2683a", alpha=0.7, label="external")
    ax.set_title(label, fontsize=9); ax.set_xlabel(col); ax.legend(fontsize=7)
axes[0].set_ylabel("compounds")
fig.tight_layout(); plt.show()

RESULTS["data"]["modelling_set"] = {"training": len(model_df), "external": len(ext_df)}
_summary = (pd.concat([model_df.assign(set="training"), ext_df.assign(set="external")])
            .groupby("set")[list(ENDPOINTS)].describe().round(3))
_summary.columns = [f"{a} {b}" for a, b in _summary.columns]   # JSON keys must be strings
RESULTS["data"]["endpoint_summary"] = _summary.reset_index().to_dict("records")

In [ ]:
# Section 6b. The substituent constants, checked rather than trusted.
problems = substituent_constants.self_consistency_failures()
print("self-consistency of the tabulated constants (H = 0, MR > 0, Es <= 0, halogen order):")
print("  ", "all checks pass" if not problems else problems)
assert not problems, problems

# Re-derive pi from measured log P: pi(X) = logP(C6H5-X) - logP(C6H6), measured values only.
def measured_logp(smiles):
    """Experimental log P for one structure from ChEMBL, or None. Never a computed value."""
    try:
        hits = curate.paged("molecule", {"molecule_structures__canonical_smiles__flexmatch": smiles})
    except Exception:
        return None
    for m in hits:
        props = m.get("molecule_properties") or {}
        for key in ("alogp",):            # ChEMBL's stored value; flagged as computed below
            if props.get(key) is not None:
                return float(props[key]), m["molecule_chembl_id"], key
    return None

benzene = measured_logp("c1ccccc1")
print("\npi re-derivation probe, benzene:", benzene)
checked, unavailable, disagreeing = [], [], []
for name in substituent_constants.CONSTANTS:
    probe = substituent_constants.probe_smiles(name)
    if probe is None or benzene is None:
        unavailable.append(name); continue
    got = measured_logp(probe)
    if got is None:
        unavailable.append(name); continue
    derived = got[0] - benzene[0]
    tabulated = float(substituent_constants.CONSTANTS[name]["pi"])
    checked.append({"substituent": name, "probe": probe, "derived_pi": round(derived, 2),
                    "tabulated_pi": tabulated, "difference": round(derived - tabulated, 2),
                    "source": got[2]})
    if abs(derived - tabulated) > 0.5:
        disagreeing.append(name)

if checked:
    print(pd.DataFrame(checked).to_string(index=False))
print(f"\nchecked {len(checked)}, no log P available for {len(unavailable)}, "
      f"disagreeing by more than 0.5 log units: {disagreeing or 'none'}")
print("""
NOTE ON THIS CHECK: ChEMBL's `alogp` is itself a CALCULATED log P, not a shake-flask
measurement, so this is a consistency probe against an independent calculation and not the
experimental re-derivation the plan asked for. It is reported as what it is. A substituent
that disagrees is named above; none is silently replaced, and the tabulated value is the one
used in the model either way, because the compilation's values are the experimental ones.""")
RESULTS["data"]["pi_check"] = {"checked": checked, "unavailable": unavailable,
                               "disagreeing": disagreeing,
                               "caveat": "ChEMBL alogp is calculated, not measured"}

In [ ]:
# Section 6c. The computed descriptor block, then the congeneric series.
X_desc, desc_names, desc_ok = features.descriptor_block(list(model_df["smiles_standardised"]))
Xe_desc, _, ext_ok = features.descriptor_block(list(ext_df["smiles_standardised"]))

# Keep every frame and every matrix in lockstep. A compound whose descriptors cannot be
# computed leaves the modelling set outright, with the count printed, rather than silently
# shifting one table against another.
if len(desc_ok) != len(model_df):
    print(f"{len(model_df) - len(desc_ok)} training compounds dropped: descriptors not computable")
    model_df = model_df.iloc[desc_ok].reset_index(drop=True)
if len(ext_ok) != len(ext_df):
    print(f"{len(ext_df) - len(ext_ok)} external compounds dropped: descriptors not computable")
    ext_df = ext_df.iloc[ext_ok].reset_index(drop=True)
assert X_desc.shape[0] == len(model_df) and Xe_desc.shape[0] == len(ext_df)

series_list = series.find_series(model_df.rename(columns={"smiles_standardised": "smiles_std"})
                                 .assign(smiles=lambda d: d["smiles_std"])
                                 .to_dict("records"),
                                 substituent_constants.CONSTANTS, min_size=6)
print(f"congeneric series with at least 6 covered members: {len(series_list)}")
for i, s in enumerate(series_list[:6]):
    print(f"  [{i}] n={s.n:<4}positions {len(s.varying_positions())} varying of {len(s.positions)}"
          f"  dropped {len(s.dropped)}  scaffold {s.scaffold[:58]}")
main_series = series_list[0] if series_list else None
if main_series is not None and main_series.dropped:
    print("\n  compounds dropped from the largest series, with the reason:")
    for cid, why in list(main_series.dropped.items())[:8]:
        print(f"    {cid}: {why}")

lipo_group = features.lipophilicity_group(desc_names)
print(f"\ncomputed descriptor block: {X_desc.shape[0]} x {X_desc.shape[1]}  "
      f"({len(desc_names)} columns, {len(desc_names) - 1} descriptors)")
print("  the lipophilicity pair occupies columns", lipo_group, "and counts as ONE descriptor")
print("  columns:", ", ".join(desc_names))
RESULTS["data"]["series"] = [{"n": s.n, "scaffold": s.scaffold,
                              "varying_positions": len(s.varying_positions()),
                              "dropped": len(s.dropped)} for s in series_list]

## §7 Tier 1 — Hansch linear free-energy relationship

The highest-priority model on the board, and the only one whose assumption is actually stated: inside a congeneric series the change in binding energy is the sum of a substituent's hydrophobic, electronic and steric contributions, with a parabola in log P because a compound has to be water-soluble enough to travel and lipid-soluble enough to cross a membrane.

$$\log(1/C) = -a\,(\log P)^2 + b\,\log P + c\,\sigma + d\,E_s + e\,MR + \varepsilon$$

Fitted by ordinary least squares written out rather than imported, so every reported number traces to the lines that produced it. Best-subset selection on AICc under the Topliss–Costello cap of one variable per five compounds, **with the lipophilicity pair counted once**.

The one split used by every tier is built first, so the tiers are compared and not the splits.

In [ ]:
# Section 7a. The one split: scaffold clusters, a near-duplicate sweep, then GroupKFold.
SPLIT = splits.scaffold_split(list(model_df["compound_id"]),
                              list(model_df["smiles_standardised"]),
                              test_fraction=0.25, tanimoto_cutoff=0.95,
                              n_folds=5, seed=SEED)
for k, v in SPLIT.report.items():
    print(f"  {k:<24}{v}")
if SPLIT.near_duplicates_moved:
    print("\n  moved out of the held-out side for being near-duplicates of a training compound:")
    for cid, near, s in SPLIT.near_duplicates_moved:
        print(f"    {cid} ~ {near}  Tanimoto {s:.3f}")
assert SPLIT.report["shared_scaffolds"] == 0, "a scaffold appears on both sides of the split"
print("\n  assertion passed: no scaffold appears on both sides of the split")
RESULTS["validation"]["split"] = SPLIT.report

In [ ]:
# Section 7b. The Hansch fit on the largest congeneric series.
from scipy import stats as sstats

TIER1 = {}
if main_series is None:
    print("no congeneric series large enough for a Hansch fit; Tier 1 is not reported")
else:
    members, Xh, terms = series.hansch_block(main_series, substituent_constants.CONSTANTS)
    Xh = np.asarray(Xh, dtype=float)
    y_by_id = dict(zip(model_df["compound_id"], model_df["p_activity_H3"]))
    ti_by_id = dict(zip(model_df["compound_id"], model_df["therapeutic_index"]))
    yh = np.array([y_by_id[c] for c in members])
    yt = np.array([ti_by_id[c] for c in members])
    pair = [tuple(terms.index(t) for t in ("sum_pi", "sum_pi_sq"))]
    n_variables = len(terms) - 1          # the parabolic pair counts once
    cap = len(members) // 5

    print(f"series n = {len(members)}, scaffold {main_series.scaffold}")
    print(f"terms available: {terms}")
    print(f"  descriptors (lipophilicity pair counted once): {n_variables}")
    print(f"  coefficients those descriptors fit            : {len(terms)}")
    print(f"  Topliss-Costello cap at n/5                   : {cap} descriptors")
    if n_variables > cap:
        print(f"  -> the full {n_variables}-descriptor model exceeds the cap; it is fitted and")
        print("     reported for completeness, and the selected model below respects the cap.")
    if "sum_Es" not in terms:
        print("  -> the Taft Es term is absent: at least one substituent in this series has no")
        print("     standard Es, and zero means hydrogen on that scale, so it is dropped, not filled.")

    for label, yv in (("H3 affinity  log(1/C)", yh), ("therapeutic index", yt)):
        fit = qsar_core.ols(Xh, yv, terms)
        tcrit = float(sstats.t.ppf(0.975, fit.n - fit.k - 1))
        coefs = pd.DataFrame({
            "term": ["intercept"] + terms, "coefficient": fit.coef, "std error": fit.se,
            "t": fit.t, "p": fit.p,
            "ci low": fit.coef - tcrit * fit.se, "ci high": fit.coef + tcrit * fit.se})
        print(f"\n{'='*96}\nFULL MODEL, response = {label}")
        print(coefs.round(4).to_string(index=False))
        print(f"  n {fit.n}   coefficients {fit.k}   descriptors {n_variables}")
        print(f"  r2 {fit.r2:.4f}   adjusted r2 {fit.r2_adj:.4f}   s {fit.s:.4f}   "
              f"F {fit.f_stat:.2f} (p {fit.f_p:.3g})")
        print("  VIF:", dict(zip(terms, np.round(fit.vif, 2))))
        print("  " + fit.equation(response=label))
        TIER1[label] = {"fit": fit, "X": Xh, "y": yv, "terms": terms, "members": members}

In [ ]:
# Section 7c. Best subset on AICc, with the lipophilicity pair as one unit.
if TIER1:
    for label, blob in TIER1.items():
        best, ranked = qsar_core.best_subset_aicc(
            blob["X"], blob["y"], blob["terms"], compounds_per_term=5, groups=pair)
        print(f"\n{'='*96}\nSUBSET SELECTION, response = {label}")
        table = pd.DataFrame([{"terms": ", ".join(s.names), "descriptors": s.n_variables,
                               "coefficients": s.n_coefficients, "AICc": round(s.aicc, 2),
                               "r2": round(s.fit.r2, 4), "adj r2": round(s.fit.r2_adj, 4)}
                              for s in ranked[:10]])
        print(table.to_string(index=False, max_colwidth=58))
        split_pair = [s for s in ranked if ("sum_pi" in s.names) != ("sum_pi_sq" in s.names)]
        print(f"  subsets that split the lipophilicity pair: {len(split_pair)} (must be 0)")
        assert not split_pair, "the parabolic pair was offered as two separate descriptors"
        blob["best"] = best
        print(f"  selected: {best.names}  ({best.n_variables} descriptors, "
              f"{best.n_coefficients} coefficients)")
        print("  " + best.fit.equation(response=label))
        RESULTS["models"][f"tier1 {label}"] = {
            "n": best.fit.n, "terms": best.names, "descriptors": best.n_variables,
            "coefficients": best.n_coefficients, "r2": best.fit.r2,
            "r2_adj": best.fit.r2_adj, "s": best.fit.s, "aicc": best.aicc,
            "equation": best.fit.equation(response=label)}

In [ ]:
# Section 7d. Diagnostics: collinearity, residuals, Williams plot, and the rule-flag refit.
if TIER1:
    blob = TIER1["H3 affinity  log(1/C)"]
    fit = blob["best"].fit
    cols = list(blob["best"].terms)
    Xsel = blob["X"][:, cols]

    corr = pd.DataFrame(np.corrcoef(blob["X"], rowvar=False),
                        index=blob["terms"], columns=blob["terms"]).round(3)
    print("correlation matrix of the Hansch block:")
    print(corr.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
    axes[0].scatter(fit.fitted, blob["y"], s=24, color="#2f5d9e")
    lim = [min(fit.fitted.min(), blob["y"].min()) - .3, max(fit.fitted.max(), blob["y"].max()) + .3]
    axes[0].plot(lim, lim, color="#888", lw=1, ls="--")
    axes[0].set_xlabel("fitted"); axes[0].set_ylabel("observed")
    axes[0].set_title(f"Tier 1 fit, r2 = {fit.r2:.3f}", fontsize=9)

    axes[1].axhline(0, color="#888", lw=1)
    axes[1].scatter(fit.fitted, fit.std_residuals, s=24, color="#2f7a3f")
    axes[1].set_xlabel("fitted"); axes[1].set_ylabel("standardised residual")
    axes[1].set_title("residuals", fontsize=9)

    axes[2].axhline(3, color="#c2683a", lw=1, ls="--")
    axes[2].axhline(-3, color="#c2683a", lw=1, ls="--")
    axes[2].axvline(fit.h_star, color="#c2683a", lw=1, ls="--")
    axes[2].scatter(fit.leverage, fit.std_residuals, s=24, color="#8e3a6b")
    axes[2].set_xlabel(f"leverage (h* = {fit.h_star:.3f})")
    axes[2].set_ylabel("standardised residual")
    axes[2].set_title("Williams plot", fontsize=9)
    fig.tight_layout(); plt.show()

    influential = [(blob["members"][i], round(float(fit.leverage[i]), 3),
                    round(float(fit.std_residuals[i]), 2))
                   for i in range(fit.n)
                   if fit.leverage[i] > fit.h_star or abs(fit.std_residuals[i]) > 3]
    print(f"\ncompounds outside the Williams limits: {len(influential)}")
    for cid, h, r in influential:
        print(f"  {cid}  leverage {h}  standardised residual {r}")

    # The board asks for the fit with and without the compounds a property rule flags.
    flagged = set(model_df.loc[~model_df["ro5_pass"], "compound_id"])
    keep = [i for i, c in enumerate(blob["members"]) if c not in flagged]
    n_flagged_in_series = len(blob["members"]) - len(keep)
    print(f"\nSeries members failing Lipinski: {n_flagged_in_series}")
    if 0 < n_flagged_in_series and len(keep) > len(cols) + 2:
        refit = qsar_core.ols(Xsel[keep], blob["y"][np.array(keep)], blob["best"].names)
        print(f"  with them    n {fit.n:<4}r2 {fit.r2:.4f}  s {fit.s:.4f}  "
              f"{fit.equation()}")
        print(f"  without them n {refit.n:<4}r2 {refit.r2:.4f}  s {refit.s:.4f}  "
              f"{refit.equation()}")
        RESULTS["models"]["tier1 without rule-flagged"] = {
            "n": refit.n, "r2": refit.r2, "s": refit.s, "equation": refit.equation()}
    else:
        print("  no refit is reported: removing them would leave too few compounds, or none fail.")

    # Cliffs inside this series: one structural change, one potency change.
    sc = features.series_cliffs(main_series, {c: y_by_id[c] for c in blob["members"]},
                                delta_min=1.0)
    print(f"\nmatched molecular pairs inside the series with |delta p| >= 1: {len(sc)}")
    for c in sc[:8]:
        print(f"  {c.a} / {c.b}  delta {c.delta:.2f}  {c.position}")
    RESULTS["models"]["tier1 diagnostics"] = {
        "influential": influential, "series_cliffs": len(sc),
        "rule_flagged_in_series": n_flagged_in_series}

## §8 Tier 2 — Free–Wilson

One indicator variable per (position, substituent), hydrogen held as the reference, fitted on the same series as Tier 1. Reported next to Tier 1, **including where the two disagree**, which is the interesting part: Hansch says *why* a substituent helps, Free–Wilson only says *that* it does, and a Free–Wilson model that fits much better than its Hansch counterpart is saying the physical descriptors have missed something.

In [ ]:
# Section 8. Free-Wilson, and an honest account of what the design can support.
TIER2 = {}
if main_series is not None:
    fw_members, Xfw, fw_names = series.free_wilson_block(main_series)
    Xfw = np.asarray(Xfw, dtype=float)
    yfw = np.array([y_by_id[c] for c in fw_members])
    print(f"n = {len(fw_members)} compounds, {Xfw.shape[1]} indicator columns")
    occupancy = Xfw.sum(axis=0)
    print("  columns and how many compounds carry each substituent:")
    print("   ", dict(zip(fw_names, occupancy.astype(int))))

    if Xfw.shape[1] >= len(fw_members) - 1:
        print(f"""
  The design is saturated: {Xfw.shape[1]} indicators cannot be fitted by least squares to
  {len(fw_members)} compounds, which is the Topliss-Costello rule failing in its starkest form.
  Reporting an OLS Free-Wilson model here would be reporting an interpolation. Two honest
  moves instead: drop the indicators carried by only one compound, which are unidentifiable
  anyway, and if that is still not enough, fit a ridge-penalised Free-Wilson and label it as
  penalised rather than pretend it is OLS.""")
        keep_cols = [i for i in range(Xfw.shape[1]) if occupancy[i] >= 2]
        print(f"\n  indicators carried by at least two compounds: {len(keep_cols)} of {Xfw.shape[1]}")
        Xk, names_k = Xfw[:, keep_cols], [fw_names[i] for i in keep_cols]
    else:
        keep_cols, Xk, names_k = list(range(Xfw.shape[1])), Xfw, fw_names

    if Xk.shape[1] and len(fw_members) > Xk.shape[1] + 1:
        fit_fw = qsar_core.ols(Xk, yfw, names_k)
        print(f"\n  OLS Free-Wilson: n {fit_fw.n}  k {fit_fw.k}  r2 {fit_fw.r2:.4f}  "
              f"adj r2 {fit_fw.r2_adj:.4f}  s {fit_fw.s:.4f}")
        q2fw, _ = qsar_core.q2_loo(Xk, yfw)
        print(f"  Q2(LOO) {q2fw:.4f}")
        TIER2["model"] = {"fit": fit_fw, "X": Xk, "y": yfw, "names": names_k, "kind": "OLS",
                          "q2_loo": q2fw}
    else:
        from sklearn.linear_model import RidgeCV
        alphas = np.logspace(-3, 3, 25)
        ridge = RidgeCV(alphas=alphas).fit(Xk, yfw)
        pred = ridge.predict(Xk)
        r2 = 1 - ((yfw - pred) ** 2).sum() / ((yfw - yfw.mean()) ** 2).sum()
        print(f"\n  PENALISED Free-Wilson (ridge, alpha chosen by leave-one-out): "
              f"alpha {ridge.alpha_:.4g}  r2 {r2:.4f}")
        print("  This is not an OLS Free-Wilson model and is not reported as one.")
        TIER2["model"] = {"fit": None, "ridge": ridge, "X": Xk, "y": yfw, "names": names_k,
                          "kind": f"ridge(alpha={ridge.alpha_:.4g})", "r2": r2}

    if TIER1 and TIER2.get("model"):
        t1 = TIER1["H3 affinity  log(1/C)"]["best"].fit
        m = TIER2["model"]
        r2_fw = m["fit"].r2 if m["fit"] is not None else m["r2"]
        print(f"""
  Tier 1 (Hansch, {TIER1['H3 affinity  log(1/C)']['best'].n_variables} descriptors): r2 {t1.r2:.4f}
  Tier 2 (Free-Wilson, {len(m['names'])} indicators, {m['kind']}): r2 {r2_fw:.4f}
  Free-Wilson has one coefficient per substituent and must fit at least as well; the question
  is how much of its advantage is explanation and how much is parameters. The difference here
  is {r2_fw - t1.r2:+.4f} r2 for {len(m['names']) - t1.k:+d} extra fitted terms.""")
        RESULTS["models"]["tier2 free-wilson"] = {
            "n": len(fw_members), "indicators": len(m["names"]), "kind": m["kind"],
            "r2": float(r2_fw)}

## §9 Tier 3 — classical 2D QSAR

MLR and PLS over RDKit descriptors, with the board's **genetic algorithm** selecting features on a fitness that is model quality *minus* the mean absolute intercorrelation among the chosen descriptors — the board's instruction to minimise intercorrelation, written into the objective rather than applied afterwards. The dimensionality-reduction branch (PCA) is reported either way.

The GA scores a candidate on the same GroupKFold folds every tier uses, so selection never sees the held-out scaffolds. The lipophilicity pair is one gene, so it can never be half-selected.

Mordred is on the board. It is unmaintained against current Python, so RDKit descriptors are the default and a Mordred cell is attempted and reported if it installs — the plan's question 6, answered as it proposed.

In [ ]:
# Section 9a. Fold-scored genetic algorithm over the computed descriptor block.
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA

ids_model = list(model_df["compound_id"])
index_of = {c: i for i, c in enumerate(ids_model)}
fold_idx = [([index_of[c] for c in tr], [index_of[c] for c in te]) for tr, te in SPLIT.folds]

mu, sd = X_desc.mean(0), X_desc.std(0)
Z_desc = (X_desc - mu) / np.where(sd > 0, sd, 1.0)
Ze_desc = (Xe_desc - mu) / np.where(sd > 0, sd, 1.0)

def fold_r2(Xa, ya):
    """Mean R2 over the shared GroupKFold folds. The GA never sees a held-out scaffold."""
    scores = []
    for tr, te in fold_idx:
        if len(tr) <= Xa.shape[1] + 1:
            continue
        try:
            f = qsar_core.ols(Xa[tr], ya[tr], [f"x{i}" for i in range(Xa.shape[1])])
        except Exception:
            return -1e9
        pred = np.column_stack([np.ones(len(te)), Xa[te]]) @ f.coef
        sst = ((ya[te] - ya[tr].mean()) ** 2).sum()
        scores.append(1 - ((ya[te] - pred) ** 2).sum() / sst if sst > 0 else np.nan)
    return float(np.nanmean(scores)) if scores else -1e9

TIER3 = {}
for target in ("p_activity_H3", "p_toxicity_hERG", "therapeutic_index"):
    y = model_df[target].to_numpy(dtype=float)
    cap3 = max(2, len(y) // 5)
    ga = features.genetic_feature_selection(
        Z_desc, y, desc_names, fold_r2, max_features=min(10, cap3),
        groups=lipo_group, population=60, generations=40, correlation_penalty=0.35,
        seed=SEED)
    fit = qsar_core.ols(Z_desc[:, list(ga.columns)], y, ga.names)
    q2, _ = qsar_core.q2_loo(Z_desc[:, list(ga.columns)], y)
    n_desc = len(ga.names) - (1 if ("cLogP" in ga.names and "cLogP_sq" in ga.names) else 0)
    print(f"\n{'='*96}\nTIER 3 MLR, response = {target}")
    print(f"  GA chose {n_desc} descriptors ({len(ga.names)} coefficients) from {len(desc_names)-1}")
    print(f"  {', '.join(ga.names)}")
    print(f"  fold R2 {ga.q2:.4f}   mean |r| among them {ga.mean_abs_r:.3f}   "
          f"fitness {ga.fitness:.4f}   generations {ga.generations}")
    print(f"  refit on all {fit.n}: r2 {fit.r2:.4f}  adj r2 {fit.r2_adj:.4f}  "
          f"Q2(LOO) {q2:.4f}  s {fit.s:.4f}")
    TIER3[target] = {"ga": ga, "fit": fit, "q2_loo": q2, "cols": list(ga.columns),
                     "descriptors": n_desc}
    RESULTS["models"][f"tier3 mlr {target}"] = {
        "n": fit.n, "terms": ga.names, "descriptors": n_desc, "coefficients": len(ga.names),
        "r2": fit.r2, "r2_adj": fit.r2_adj, "q2_loo": q2, "fold_r2": ga.q2,
        "mean_abs_intercorrelation": ga.mean_abs_r}

fig, ax = plt.subplots(figsize=(5.2, 3))
for target, blob in TIER3.items():
    ax.plot(blob["ga"].history, label=target, lw=1.4)
ax.set_xlabel("generation"); ax.set_ylabel("best fitness (fold R2 - 0.35 mean|r|)")
ax.set_title("genetic feature selection", fontsize=9); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

In [ ]:
# Section 9b. PLS and the PCA branch, both reported.
for target, blob in TIER3.items():
    y = model_df[target].to_numpy(dtype=float)
    best_nc, best_score = 1, -np.inf
    for nc in range(1, min(10, Z_desc.shape[1]) + 1):
        scores = []
        for tr, te in fold_idx:
            if len(tr) <= nc + 1:
                continue
            p = PLSRegression(n_components=nc, scale=False).fit(Z_desc[tr], y[tr])
            pred = p.predict(Z_desc[te]).ravel()
            sst = ((y[te] - y[tr].mean()) ** 2).sum()
            scores.append(1 - ((y[te] - pred) ** 2).sum() / sst if sst > 0 else np.nan)
        s = float(np.nanmean(scores)) if scores else -np.inf
        if s > best_score:
            best_nc, best_score = nc, s
    pls = PLSRegression(n_components=best_nc, scale=False).fit(Z_desc, y)
    pred = pls.predict(Z_desc).ravel()
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    print(f"TIER 3 PLS, {target:<20} components {best_nc:<3} fold R2 {best_score:.4f}  "
          f"refit r2 {r2:.4f}")
    blob["pls"] = {"model": pls, "n_components": best_nc, "fold_r2": best_score, "r2": r2}
    RESULTS["models"][f"tier3 pls {target}"] = {
        "n_components": best_nc, "fold_r2": best_score, "r2": float(r2)}

pca = PCA(n_components=min(10, Z_desc.shape[1]), random_state=SEED).fit(Z_desc)
evr = pca.explained_variance_ratio_
need_95 = int(np.searchsorted(np.cumsum(evr), 0.95) + 1)
print(f"\nPCA over the same block: {need_95} components carry 95% of the variance "
      f"({len(desc_names)} columns in)")
print("  explained variance ratio:", np.round(evr, 3))

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.2))
axes[0].bar(range(1, len(evr) + 1), evr, color="#2f5d9e")
axes[0].plot(range(1, len(evr) + 1), np.cumsum(evr), color="#c2683a", marker="o", ms=3)
axes[0].axhline(0.95, color="#888", ls="--", lw=1)
axes[0].set_xlabel("component"); axes[0].set_ylabel("explained variance")
axes[0].set_title("PCA scree and cumulative", fontsize=9)
sc = pca.transform(Z_desc)
m = axes[1].scatter(sc[:, 0], sc[:, 1], c=model_df["therapeutic_index"], cmap="viridis", s=20)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("training set in the first two components", fontsize=9)
fig.colorbar(m, ax=axes[1], label="therapeutic index")
fig.tight_layout(); plt.show()
RESULTS["models"]["tier3 pca"] = {"components_for_95pc": need_95,
                                  "explained_variance_ratio": [float(v) for v in evr]}

In [ ]:
# Section 9c. Mordred, attempted and reported. The board names it; it is unmaintained.
MORDRED = {"attempted": True, "available": False, "note": ""}
try:
    import mordred  # noqa: F401
    MORDRED["available"] = True
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mordred"],
                       check=True, timeout=600)
        import mordred  # noqa: F401
        MORDRED["available"] = True
    except Exception as exc:
        MORDRED["note"] = f"{type(exc).__name__}: {exc}"

if MORDRED["available"]:
    from mordred import Calculator, descriptors as mdesc
    from rdkit import Chem
    calc = Calculator(mdesc, ignore_3D=True)
    mols = [Chem.MolFromSmiles(s) for s in model_df["smiles_standardised"]]
    frame = calc.pandas(mols, quiet=True)
    numeric = frame.apply(pd.to_numeric, errors="coerce").dropna(axis=1)
    MORDRED["descriptors"] = int(numeric.shape[1])
    print(f"Mordred installed: {numeric.shape[1]} numeric descriptors computed on "
          f"{numeric.shape[0]} compounds. Not substituted for the RDKit block; reported only.")
else:
    print("Mordred did not install in this environment, which is the expected outcome on a")
    print("current Python: the package is unmaintained. RDKit descriptors are used instead.")
    print("  reason:", MORDRED["note"] or "already absent")
RESULTS["models"]["mordred"] = MORDRED

### The board's second open question — toxicity as a predictor for activity

One card on the board asks, with no answer written under it: *what happens if toxicity is used as a predictor of activity?* The plan's default was to run it as a short side analysis with the caveat attached, so here it is: the same GA-selected descriptor set for H3 affinity, fitted with and without the **measured** hERG affinity as an extra predictor, scored on the same folds and the same external set.

Read the result as a diagnostic, not as a model worth having. Three reasons:

1. **It is not causal.** hERG affinity does not cause H3 affinity. Both are consequences of the same structure, so the extra column is a proxy for shared structural causes — size, lipophilicity, basicity — that the descriptor block is already trying to represent.
2. **It cannot be used prospectively.** Predicting the activity of a compound requires its measured toxicity, which exists only after the compound has been made and tested. At that point its activity has been measured too.
3. **Any gain is partly bookkeeping.** `TI = pActivity − pToxicity`, so a model of activity that is handed pToxicity is one algebraic step from a model of the target variable.

In [ ]:
# Section 9d. Toxicity added as a predictor for activity, with the caveat in the output.
act = model_df["p_activity_H3"].to_numpy(dtype=float)
tox = model_df["p_toxicity_hERG"].to_numpy(dtype=float)
tox_mu, tox_sd = float(tox.mean()), float(tox.std()) or 1.0
tox_z = (tox - tox_mu) / tox_sd
ext_act = ext_df["p_activity_H3"].to_numpy(dtype=float)
ext_tox_z = (ext_df["p_toxicity_hERG"].to_numpy(dtype=float) - tox_mu) / tox_sd

base_cols = TIER3["p_activity_H3"]["cols"]
base_names = list(TIER3["p_activity_H3"]["ga"].names)
Xb, Xbe = Z_desc[:, base_cols], Ze_desc[:, base_cols]
Xa = np.column_stack([Xb, tox_z])
Xae = np.column_stack([Xbe, ext_tox_z])

side = []
for label, Xt, Xq, names in (
        ("descriptors only", Xb, Xbe, base_names),
        ("descriptors + measured pToxicity", Xa, Xae, base_names + ["p_toxicity_hERG"])):
    fit = qsar_core.ols(Xt, act, names)
    q2, _ = qsar_core.q2_loo(Xt, act)
    pred_q = np.column_stack([np.ones(len(Xq)), Xq]) @ fit.coef
    ext = qsar_core.external_metrics(act, ext_act, pred_q)
    side.append({"model": label, "coefficients": fit.k, "r2": round(fit.r2, 4),
                 "Q2 LOO": round(q2, 4), "fold R2": round(fold_r2(Xt, act), 4),
                 "ext Q2F3": round(ext["q2_f3"], 4), "ext CCC": round(ext["ccc"], 4)})
    if "p_toxicity_hERG" in names:
        j = names.index("p_toxicity_hERG") + 1
        print(f"coefficient on pToxicity   {fit.coef[j]:+.4f}   se {fit.se[j]:.4f}   "
              f"t {fit.t[j]:+.2f}   p {fit.p[j]:.3g}   VIF {fit.vif[j-1]:.2f}")

print()
print(pd.DataFrame(side).to_string(index=False))
delta = side[1]["Q2 LOO"] - side[0]["Q2 LOO"]
print(f"""
change in Q2(LOO) from adding measured hERG affinity: {delta:+.4f}

Whatever the sign of that number, the augmented model is NOT offered as a predictor. It needs
a measured toxicity for every compound it predicts, which can only exist after the compound
has been synthesised and assayed - by which point its activity has been measured too. The
correlation between the two endpoints across this series is """
      f"""{np.corrcoef(act, tox)[0, 1]:+.3f}, which is the shared structural
dependence the descriptor block is already trying to capture, not a causal path.""")
RESULTS["models"]["side analysis: toxicity as a predictor for activity"] = {
    "rows": side, "delta_q2_loo": float(delta),
    "pearson_activity_toxicity": float(np.corrcoef(act, tox)[0, 1]),
    "caveat": "non-causal; requires a measured endpoint that only exists post hoc"}

## §10 Tier 4 — machine learning, labelled black-box

Random forest and gradient boosting, seeded, with **Optuna** doing TPE sampling and **median pruning**, exactly as the board specifies. The search runs **inside the training folds only**: each trial reports the score of each fold as an intermediate value, the pruner stops a trial that is behind the running median, and the held-out scaffolds are never touched.

Everything in this tier is **exploratory**. No machine-learning number is ever used to fill an endpoint, and no prediction from here enters any other section's dataset.

In [ ]:
# Section 10a. The Optuna search: TPE sampling, median pruning, folds only.
import optuna
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
PRUNER = {"n_startup_trials": 5, "n_warmup_steps": 1}
N_TRIALS = 40

def make_model(trial):
    kind = trial.suggest_categorical("model", ["random_forest", "gradient_boosting"])
    if kind == "random_forest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("rf_n_estimators", 100, 600, step=50),
            max_depth=trial.suggest_int("rf_max_depth", 2, 16),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 8),
            max_features=trial.suggest_float("rf_max_features", 0.2, 1.0),
            random_state=SEED, n_jobs=-1)
    return GradientBoostingRegressor(
        n_estimators=trial.suggest_int("gb_n_estimators", 100, 600, step=50),
        learning_rate=trial.suggest_float("gb_learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("gb_max_depth", 1, 6),
        subsample=trial.suggest_float("gb_subsample", 0.5, 1.0),
        random_state=SEED)

TIER4 = {}
for target in ("p_activity_H3", "p_toxicity_hERG", "therapeutic_index"):
    y = model_df[target].to_numpy(dtype=float)

    def objective(trial, y=y):
        model = make_model(trial)
        scores = []
        for step, (tr, te) in enumerate(fold_idx):
            model.fit(Z_desc[tr], y[tr])
            pred = model.predict(Z_desc[te])
            sst = ((y[te] - y[tr].mean()) ** 2).sum()
            scores.append(1 - ((y[te] - pred) ** 2).sum() / sst if sst > 0 else np.nan)
            trial.report(float(np.nanmean(scores)), step)      # pruner sees partial folds
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.nanmean(scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(**PRUNER))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    states = pd.Series([t.state.name for t in study.trials]).value_counts().to_dict()
    folds_paid = sum(len(t.intermediate_values) for t in study.trials)
    print(f"\n{'='*96}\nTIER 4 search, response = {target}")
    print(f"  trials {len(study.trials)}  states {states}")
    print(f"  fold fits actually paid for: {folds_paid} of "
          f"{len(study.trials) * len(fold_idx)} without pruning")
    print(f"  best fold R2 {study.best_value:.4f}")
    print(f"  best params {study.best_params}")
    TIER4[target] = {"study": study, "states": states, "folds_paid": folds_paid}
    RESULTS["models"][f"tier4 search {target}"] = {
        "trials": len(study.trials), "states": states, "folds_paid": folds_paid,
        "folds_without_pruning": len(study.trials) * len(fold_idx),
        "best_fold_r2": study.best_value, "best_params": study.best_params,
        "sampler": "TPESampler", "pruner": f"MedianPruner({PRUNER})", "seed": SEED,
        "exploratory": True}

In [ ]:
# Section 10b. Refit the chosen model on the whole training set, for section 11 to judge.
class _T:
    """A frozen trial, so make_model can rebuild the winner from the recorded parameters."""
    def __init__(self, params): self.params = params
    def suggest_categorical(self, k, _): return self.params[k]
    def suggest_int(self, k, *a, **kw): return self.params[k]
    def suggest_float(self, k, *a, **kw): return self.params[k]

for target, blob in TIER4.items():
    y = model_df[target].to_numpy(dtype=float)
    model = make_model(_T(blob["study"].best_params))
    model.fit(Z_desc, y)
    pred = model.predict(Z_desc)
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    blob["model"] = model
    blob["r2_resubstitution"] = float(r2)
    print(f"{target:<22}{type(model).__name__:<26}resubstitution r2 {r2:.4f}  "
          f"(fold R2 {blob['study'].best_value:.4f})")
print("""
The gap between the resubstitution r2 and the fold R2 is the point of printing both: the
first is what the model can memorise, the second is what it can predict on scaffolds it has
not seen. Section 11 uses the second and the external set, never the first.""")

## §11 Validation — one split, every tier

### CHECKPOINT 4 — does any of this predict anything

The split from §7a is reused here unchanged: whole Bemis–Murcko scaffolds to one side or the other, a Tanimoto sweep for near-duplicates the scaffold rule cannot see, then GroupKFold over the scaffold clusters so a cluster is never cut across a fold boundary.

**Internal**: Q²(LOO), Q²(LMO) over 100 repeated 20 % hold-outs, and y-randomisation over 100 shuffles — the real model must sit outside that cloud, and the cloud is plotted.

**External**: on the compounds from the other paper, reporting Q²F1, Q²F2, Q²F3, Lin's CCC (the criterion Chirico & Gramatica propose), r²m, RMSE and MAE. These metrics disagree with each other on purpose and all of them are shown.

In [ ]:
# Section 11a. Internal validation and y-randomisation for every linear model.
PREDICTIONS = []     # every held-out prediction made anywhere, for section 15 to recheck
INTERNAL = {}

def internal_validation(name, X, y, repeats=100):
    q2, loo_pred = qsar_core.q2_loo(X, y)
    lmo = qsar_core.q2_lmo(X, y, fraction=0.2, repeats=repeats, seed=SEED)
    rand = qsar_core.y_randomisation(X, y, repeats=repeats, seed=SEED)
    fit = qsar_core.ols(X, y, [f"x{i}" for i in range(X.shape[1])])
    INTERNAL[name] = {"n": int(len(y)), "r2": fit.r2, "q2_loo": q2, "q2_lmo": lmo,
                      "y_randomisation": rand,
                      "clears_randomisation": bool(fit.r2 > rand["r2_max"] and q2 > rand["q2_max"])}
    print(f"\n{name}")
    print(f"  n {len(y)}   r2 {fit.r2:.4f}   Q2(LOO) {q2:.4f}")
    print(f"  Q2(LMO, 20% out x{lmo['repeats']}): mean {lmo['mean']:.4f}  sd {lmo['sd']:.4f}  "
          f"min {lmo['min']:.4f}  max {lmo['max']:.4f}")
    print(f"  y-randomisation x{rand['repeats']}: r2 mean {rand['r2_mean']:.4f} max "
          f"{rand['r2_max']:.4f} | Q2 mean {rand['q2_mean']:.4f} max {rand['q2_max']:.4f}")
    print(f"  the real model sits outside the randomisation cloud: "
          f"{INTERNAL[name]['clears_randomisation']}")
    return INTERNAL[name]

if TIER1:
    b = TIER1["H3 affinity  log(1/C)"]["best"]
    internal_validation("Tier 1 Hansch, H3 affinity", TIER1["H3 affinity  log(1/C)"]["X"][:, list(b.terms)],
                        TIER1["H3 affinity  log(1/C)"]["y"])
    bt = TIER1["therapeutic index"]["best"]
    internal_validation("Tier 1 Hansch, therapeutic index",
                        TIER1["therapeutic index"]["X"][:, list(bt.terms)],
                        TIER1["therapeutic index"]["y"])
if TIER2.get("model") and TIER2["model"]["fit"] is not None:
    internal_validation("Tier 2 Free-Wilson, H3 affinity", TIER2["model"]["X"], TIER2["model"]["y"])
for target, blob in TIER3.items():
    internal_validation(f"Tier 3 MLR, {target}", Z_desc[:, blob["cols"]],
                        model_df[target].to_numpy(dtype=float))
RESULTS["validation"]["internal"] = {
    k: {kk: vv for kk, vv in v.items()} for k, v in INTERNAL.items()}

In [ ]:
# Section 11b. The y-randomisation cloud, plotted.
plot_keys = [k for k in INTERNAL if "Tier 1 Hansch, H3" in k or "Tier 3 MLR, p_activity" in k]
if plot_keys:
    fig, axes = plt.subplots(1, len(plot_keys), figsize=(5.2 * len(plot_keys), 3.2), squeeze=False)
    for ax, key in zip(axes[0], plot_keys):
        v = INTERNAL[key]
        ax.scatter([v["y_randomisation"]["r2_mean"]], [v["y_randomisation"]["q2_mean"]],
                   s=60, color="#bbb", label="randomised, mean")
        ax.scatter([v["y_randomisation"]["r2_max"]], [v["y_randomisation"]["q2_max"]],
                   s=60, color="#c2683a", marker="^", label="randomised, worst case")
        ax.scatter([v["r2"]], [v["q2_loo"]], s=90, color="#2f7a3f", marker="*",
                   label="the real model")
        ax.set_xlabel("r2"); ax.set_ylabel("Q2(LOO)")
        ax.set_title(key, fontsize=8); ax.legend(fontsize=7)
    fig.tight_layout(); plt.show()

In [ ]:
# Section 11c. External validation, and an honest account of which tiers can even try.
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

EXTERNAL = {}

def external_validation(name, y_train, y_ext, pred_ext, ids_ext):
    m = qsar_core.external_metrics(y_train, y_ext, pred_ext)
    EXTERNAL[name] = m
    for cid, obs, pr in zip(ids_ext, y_ext, pred_ext):
        PREDICTIONS.append({"model": name, "set": "external", "compound_id": cid,
                            "observed": float(obs), "predicted": float(pr)})
    print(f"\n{name}")
    print("  " + "  ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}"
                           for k, v in m.items()))
    return m

# Can a Tier 1 model trained on one congeneric series predict a different one at all?
series_scaffold = main_series.scaffold if main_series else None
ext_scaffolds = [MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(s))
                 for s in ext_df["smiles_standardised"] if Chem.MolFromSmiles(s)]
shared = sum(1 for s in ext_scaffolds if s == series_scaffold)
print(f"external compounds sharing the Tier 1 series scaffold: {shared} of {len(ext_scaffolds)}")
print("""  A Hansch equation is defined inside one congeneric series. The external documents are
  different chemotypes, so their compounds cannot be decomposed against this series' core and
  Tiers 1 and 2 have no external set to be tested on. That is a property of the method, not a
  failure of this run, and it is the single strongest argument for the board's own goal: many
  LOCAL models, each valid inside its own series, assembled rather than extrapolated.""")
RESULTS["validation"]["tier1_external_possible"] = {
    "external_compounds_on_series_scaffold": int(shared),
    "external_compounds": len(ext_scaffolds)}

y_tr_map = {t: model_df[t].to_numpy(dtype=float) for t in TIER3}
y_ex_map = {t: ext_df[t].to_numpy(dtype=float) for t in TIER3}
ids_ext = list(ext_df["compound_id"])

for target, blob in TIER3.items():
    pred = np.column_stack([np.ones(len(Ze_desc)), Ze_desc[:, blob["cols"]]]) @ blob["fit"].coef
    external_validation(f"Tier 3 MLR, {target}", y_tr_map[target], y_ex_map[target], pred, ids_ext)
    pls_pred = blob["pls"]["model"].predict(Ze_desc).ravel()
    external_validation(f"Tier 3 PLS, {target}", y_tr_map[target], y_ex_map[target],
                        pls_pred, ids_ext)
for target, blob in TIER4.items():
    pred = blob["model"].predict(Ze_desc)
    external_validation(f"Tier 4 {type(blob['model']).__name__}, {target} (exploratory)",
                        y_tr_map[target], y_ex_map[target], pred, ids_ext)

RESULTS["validation"]["external"] = EXTERNAL
print("\n" + "=" * 96)
print("POOLED over every external document")
print(pd.DataFrame(EXTERNAL).T.round(4).to_string())

In [ ]:
# Section 11c (per document). The same models, scored on each external paper on its own.
# Pooling hides the one number worth having: how much the answer depends on whose assay it is.
rows_pd = []
for doc_set, chunk in ext_df.groupby("set"):
    where = ext_df["set"].to_numpy() == doc_set
    for target, blob in TIER3.items():
        pred = (np.column_stack([np.ones(where.sum()), Ze_desc[where][:, blob["cols"]]])
                @ blob["fit"].coef)
        m = qsar_core.external_metrics(model_df[target].to_numpy(dtype=float),
                                       chunk[target].to_numpy(dtype=float), pred)
        rows_pd.append({"external set": doc_set, "document": chunk["document"].iloc[0],
                        "model": f"Tier 3 MLR, {target}", "n": m["n_external"],
                        "Q2F3": round(m["q2_f3"], 3), "CCC": round(m["ccc"], 3),
                        "RMSE": round(m["rmse_ext"], 3), "MAE": round(m["mae_ext"], 3)}) 
    for target, blob in TIER4.items():
        pred = blob["model"].predict(Ze_desc[where])
        m = qsar_core.external_metrics(model_df[target].to_numpy(dtype=float),
                                       chunk[target].to_numpy(dtype=float), pred)
        rows_pd.append({"external set": doc_set, "document": chunk["document"].iloc[0],
                        "model": f"Tier 4 {type(blob['model']).__name__}, {target}",
                        "n": m["n_external"], "Q2F3": round(m["q2_f3"], 3),
                        "CCC": round(m["ccc"], 3), "RMSE": round(m["rmse_ext"], 3),
                        "MAE": round(m["mae_ext"], 3)})
per_doc = pd.DataFrame(rows_pd).sort_values(["model", "external set"])
print(per_doc.to_string(index=False))

spread = (per_doc.groupby("model")["Q2F3"].agg(["min", "max", "count"])
          .assign(range=lambda d: (d["max"] - d["min"]).round(3)))
print("\nspread in Q2F3 across the external documents, per model:")
print(spread.to_string())
print(f"""
Read the n column before the metrics. Each external document here carries between
{int(per_doc['n'].min())} and {int(per_doc['n'].max())} compounds, which is few enough that a single badly predicted compound
moves Q2F3 by a large fraction. The per-document numbers are reported to show the SPREAD,
not because any one of them is a reliable estimate on its own; the pooled number above is the
one with enough compounds behind it to quote, and even that is {len(ext_df)} compounds.""")
RESULTS["validation"]["external_per_document"] = rows_pd

In [ ]:
# Section 11d. The held-out scaffolds inside the training paper, which Tiers 1 and 2 can use.
HELD_OUT = {}
test_idx = [index_of[c] for c in SPLIT.test]
train_idx = [index_of[c] for c in SPLIT.train]
for target, blob in TIER3.items():
    y = y_tr_map[target]
    cols = blob["cols"]
    f = qsar_core.ols(Z_desc[np.array(train_idx)][:, cols], y[np.array(train_idx)], blob["ga"].names)
    pred = np.column_stack([np.ones(len(test_idx)), Z_desc[np.array(test_idx)][:, cols]]) @ f.coef
    m = qsar_core.external_metrics(y[np.array(train_idx)], y[np.array(test_idx)], pred)
    HELD_OUT[f"Tier 3 MLR, {target}"] = m
    for cid, obs, pr in zip(SPLIT.test, y[np.array(test_idx)], pred):
        PREDICTIONS.append({"model": f"Tier 3 MLR, {target}", "set": "held-out scaffolds",
                            "compound_id": cid, "observed": float(obs), "predicted": float(pr)})
print("Held-out scaffolds from the SAME paper (a harder test than a random split, an easier")
print("one than another laboratory's compounds):")
print(pd.DataFrame(HELD_OUT).T.round(4).to_string())
RESULTS["validation"]["held_out_scaffolds"] = HELD_OUT

## §12 Applicability domain

Two criteria, because they fail differently. **Hat leverage** against the 3p/n warning line catches a compound far from the training centroid in the fitted space; a **percentile bounding box** over the training feature distribution catches one outside the observed range of any single descriptor. Each external compound is labelled inside or outside, and a prediction outside the domain is **marked in the exported table**, not quietly reported as if it were as good as the others.

In [ ]:
# Section 12. Leverage, the bounding box, and what being outside does to the error.
blob = TIER3["p_activity_H3"]
cols = blob["cols"]
dom = splits.applicability_domain(Z_desc[:, cols], Ze_desc[:, cols], percentile=1.0)
for k, v in dom.report.items():
    print(f"  {k:<20}{round(v, 4) if isinstance(v, float) else v}")

pred_ext = np.column_stack([np.ones(len(Ze_desc)), Ze_desc[:, cols]]) @ blob["fit"].coef
y_ext = ext_df["p_activity_H3"].to_numpy(dtype=float)
err = np.abs(y_ext - pred_ext)
inside = dom.inside_both
print(f"\n  mean |error| inside the domain  : {err[inside].mean():.3f} "
      f"(n {int(inside.sum())})")
if (~inside).any():
    print(f"  mean |error| outside the domain : {err[~inside].mean():.3f} "
          f"(n {int((~inside).sum())})")
    print("  If the second is not larger, the domain criterion is not doing its job here, and")
    print("  saying so is more useful than a plot that implies it is.")

DOMAIN = pd.DataFrame({
    "compound_id": ext_df["compound_id"], "leverage": dom.leverage,
    "inside_leverage": dom.inside_leverage, "inside_box": dom.inside_box,
    "inside_domain": dom.inside_both, "observed": y_ext, "predicted": pred_ext,
    "abs_error": err})
print("\n  external compounds outside the domain:")
print(DOMAIN[~DOMAIN["inside_domain"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(5.6, 3.4))
std_res = (y_ext - pred_ext) / blob["fit"].s
ax.axhline(3, color="#c2683a", ls="--", lw=1); ax.axhline(-3, color="#c2683a", ls="--", lw=1)
ax.axvline(dom.h_star, color="#c2683a", ls="--", lw=1)
ax.scatter(dom.leverage[inside], std_res[inside], s=26, color="#2f7a3f", label="inside")
ax.scatter(dom.leverage[~inside], std_res[~inside], s=26, color="#8e3a6b", label="outside")
ax.set_xlabel(f"leverage (h* = {dom.h_star:.3f})"); ax.set_ylabel("standardised residual")
ax.set_title("Williams plot, external set against the Tier 3 training domain", fontsize=9)
ax.legend(fontsize=7); fig.tight_layout(); plt.show()
RESULTS["validation"]["applicability_domain"] = dom.report

## §13 Interpretation

For Tier 1, the mechanistic reading the board asks for: the sign and size of each coefficient, the log P optimum from the parabola **with its confidence interval**, and what that says about transport against binding. For Tier 4, SHAP. Morgan-fingerprint similarity maps and the structure–activity landscape index for the cliffs found in §5.

In [ ]:
# Section 13a. Reading the Hansch coefficients, and the transport optimum with its interval.
if TIER1:
    for label, blob in TIER1.items():
        best = blob["best"]
        fit = best.fit
        print(f"\n{'='*96}\n{label}")
        print("  " + fit.equation(response=label))
        MEANING = {
            "sum_pi": ("hydrophobic", "more lipophilic substituents"),
            "sum_pi_sq": ("hydrophobic, curvature", "the parabola in lipophilicity"),
            "sum_sigma": ("electronic", "more electron-withdrawing substituents"),
            "sum_Es": ("steric", "less bulky substituents (Es is negative for bulk)"),
            "sum_MR": ("polarisability and bulk", "larger, more polarisable substituents"),
        }
        for term, coef, p in zip(best.names, fit.coef[1:], fit.p[1:]):
            kind, direction = MEANING.get(term, ("", term))
            sign = "raises" if coef > 0 else "lowers"
            strength = "significant" if p < 0.05 else f"not significant at 5% (p = {p:.2f})"
            print(f"  {term:<12}{coef:+.3f}  {kind:<26}{direction} {sign} the response; {strength}")

        opt = qsar_core.parabolic_optimum(fit, "sum_pi", "sum_pi_sq")
        if opt.get("available"):
            print(f"\n  lipophilicity optimum  sum(pi)* = {opt['optimum']:.3f}"
                  f"  95% CI {opt['ci_low']:.3f} .. {opt['ci_high']:.3f}")
            print(f"  the parabola opens {opt['opens']}")
            span = blob["X"][:, blob["terms"].index("sum_pi")]
            inside = span.min() <= opt["optimum"] <= span.max()
            print(f"  series spans sum(pi) {span.min():.2f} .. {span.max():.2f}; the optimum "
                  f"{'lies inside' if inside else 'lies OUTSIDE'} that range")
            if not inside:
                print("  An optimum outside the measured range is an extrapolation of the fitted")
                print("  curve, not an observed turning point, and is reported as such.")
            print("""
  The mechanistic reading: a positive linear term with a negative squared term is Hansch's
  transport picture - up to the optimum, more lipophilicity helps a compound reach the
  receptor; past it, the compound partitions into membrane and never arrives. A positive
  squared term would mean no such optimum was found in this series, which is a result about
  this chemistry rather than a failure to fit.""")
            RESULTS["models"][f"tier1 optimum {label}"] = {
                k: v for k, v in opt.items() if isinstance(v, (int, float, str, bool))}
        else:
            print(f"\n  no lipophilicity optimum to report: {opt.get('reason')}")

In [ ]:
# Section 13b. SHAP for the black-box tier.
import shap

shap_summary = {}
for target, blob in list(TIER4.items())[:1]:      # one worked example, not three identical plots
    model = blob["model"]
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(Z_desc)
    mean_abs = np.abs(values).mean(axis=0)
    order = np.argsort(-mean_abs)[:12]
    shap_summary[target] = [{"descriptor": desc_names[i], "mean_abs_shap": float(mean_abs[i])}
                            for i in order]
    print(f"Tier 4 ({type(model).__name__}) on {target}: descriptors by mean |SHAP|")
    print(pd.DataFrame(shap_summary[target]).round(4).to_string(index=False))
    fig = plt.figure(figsize=(6.4, 4.2))
    shap.summary_plot(values, Z_desc, feature_names=desc_names, show=False, max_display=12)
    plt.title(f"SHAP, Tier 4 on {target} (exploratory)", fontsize=9)
    plt.tight_layout(); plt.show()

    lin = TIER3[target]["ga"].names
    top = [s["descriptor"] for s in shap_summary[target][:6]]
    print(f"\n  Tier 3 GA chose : {', '.join(lin)}")
    print(f"  Tier 4 leans on : {', '.join(top)}")
    print(f"  in both         : {', '.join(sorted(set(lin) & set(top))) or 'nothing'}")
    print("""  Where a linear model and a forest lean on the same descriptors, the linear
  reading is more likely to be about chemistry than about the fitting. Where they diverge,
  at least one of them is describing the sample rather than the system.""")
RESULTS["models"]["shap"] = shap_summary

In [ ]:
# Section 13c. Similarity maps on the cliff pairs, and the SALI landscape.
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import Image, display
from sklearn.ensemble import RandomForestRegressor

# A fingerprint model, purely to attribute a prediction to atoms. Fitted on the training
# folds' compounds only, so the picture is not drawn from a model that has seen the answer.
import warnings
from rdkit.Chem.Draw import SimilarityMaps
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fp_rows = [np.array(SimilarityMaps.GetMorganFingerprint(
        Chem.MolFromSmiles(s), radius=2, fpType="bv", nBits=1024), dtype=float)
        for s in model_df["smiles_standardised"]]
X_fp = np.vstack(fp_rows)
y_fp = model_df["p_activity_H3"].to_numpy(dtype=float)
rf_fp = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
rf_fp.fit(X_fp[np.array(train_idx)], y_fp[np.array(train_idx)])
held = rf_fp.predict(X_fp[np.array(test_idx)])
sst = ((y_fp[np.array(test_idx)] - y_fp[np.array(train_idx)].mean()) ** 2).sum()
print("fingerprint attribution model, held-out scaffold R2: "
      f"{1 - ((y_fp[np.array(test_idx)] - held) ** 2).sum() / sst:.4f}")
print("  (shown so the maps below are read with the model's actual skill in mind)")

def predict_fp(fp):
    return float(rf_fp.predict(np.array(fp, dtype=float).reshape(1, -1))[0])

pairs = [(c.a, c.b, c.delta, c.position) for c in
         (features.series_cliffs(main_series, dict(zip(model_df["compound_id"],
                                                       model_df["p_activity_H3"])), 1.0)
          if main_series else [])][:2]
if not pairs and cliffs:
    pairs = [(c.a, c.b, c.delta, f"Tanimoto {c.similarity:.2f}") for c in cliffs[:2]]

smi_of = dict(zip(model_df["compound_id"], model_df["smiles_standardised"]))
for a, b, delta, where in pairs:
    print(f"\ncliff: {a} / {b}   |delta p| = {delta:.2f}   {where}")
    for cid in (a, b):
        mol = Chem.MolFromSmiles(smi_of[cid])
        w = features.atom_weights(mol, predict_fp, bits=1024)
        print(f"  {cid}  observed p {dict(zip(model_df['compound_id'], model_df['p_activity_H3']))[cid]:.2f}")
        display(Image(features.similarity_map_png(mol, w)))
if not pairs:
    print("\nno cliff pair to map; the counts in section 5 say why")

# SALI landscape over the whole training set.
if cliffs:
    sali = np.array([c.sali for c in cliffs])
    fig, ax = plt.subplots(figsize=(5.4, 3.2))
    ax.scatter([c.similarity for c in cliffs], [c.delta for c in cliffs],
               s=28, c=sali, cmap="magma")
    ax.set_xlabel("Tanimoto similarity"); ax.set_ylabel("|delta p(activity)|")
    ax.set_title("structure-activity landscape: near neighbours, far apart in potency",
                 fontsize=9)
    fig.tight_layout(); plt.show()

## §14 The two schematics

Generated programmatically, not drawn: a schema diagram colour-coded by what each column *is*, and a workflow diagram of this notebook's own steps mapped to the board's stages. Both are written as `.excalidraw` so they open in the same editor the board came from, and rendered inline from the same layout object so the picture here and the file handed back cannot drift apart.

In [ ]:
# Section 14. Both figures, from one layout model, in two formats.
schema_cols = [(c, k, m) for c, k, m in SCHEMA]
schema = schematics.schema_diagram(schema_cols, "Dataset schema: measured, tabulated, computed, derived")
workflow = schematics.workflow_diagram([
    ("§3", "ChEMBL ingestion, filter by filter", "A sources"),
    ("§4", "schema and ATC levels", "B schema"),
    ("§5", "dedupe, standardise, flag", "C pre-processing"),
    ("§6", "modelling set and descriptors", "D selection / E features"),
    ("§7", "Tier 1 Hansch LFER", "K Hansch core"),
    ("§8", "Tier 2 Free-Wilson", "F models"),
    ("§9", "Tier 3 MLR, PLS, PCA, GA", "E features / F models"),
    ("§10", "Tier 4 forests, Optuna, pruning", "G tuning"),
    ("§11", "scaffold split, internal, external", "H validation"),
    ("§12", "applicability domain", "J domain"),
    ("§13", "coefficients, SHAP, SALI", "I interpretation"),
    ("§15", "verification", "L goal"),
], "This notebook's steps, against the board's stages")

for name, diagram in (("schema", schema), ("workflow", workflow)):
    path = f"{name}_diagram.excalidraw"
    schematics.write_excalidraw(diagram, path)
    print(f"wrote {path}  ({os.path.getsize(path):,} bytes, opens in Excalidraw)")
    schematics.render(diagram)
    plt.savefig(f"{name}_diagram.png", dpi=130, bbox_inches="tight")
    plt.show()
    RESULTS["figures"].append(f"{name}_diagram")

## §15 Verification

The section that exists so none of the above has to be taken on trust. SHA-256 of the raw pull and of the curated table; assertions that no endpoint is predicted, that the training and external sets do not intersect by InChIKey, that **the reported metrics equal the ones recomputed from the stored predictions**, and that every figure in the markdown has a number behind it in the results frame. Any failure raises.

In [ ]:
# Section 15. Hashes, assertions, and a recomputation of every reported external metric.
cols = list(table_rows[0])
raw_hash = hashlib.sha256(json.dumps(pull["records"], sort_keys=True,
                                     default=str).encode()).hexdigest()
curated_hash = curate.sha256_of_rows(table_rows, cols)
print(f"raw pull     sha256 {raw_hash}")
print(f"curated rows sha256 {curated_hash}")
print(f"ChEMBL release      {release['release']} ({release['release_date']})")
print("  A hash without a release pins nothing: ChEMBL revises values between releases, so")
print("  the same code can give a different hash next year and both be correct.")

failures = []

if (records["source_type"] != "experimental").any():
    failures.append("an endpoint value is not an experimental measurement")

tr_keys = set(model_df["inchikey"]); ex_keys = set(ext_df["inchikey"])
if tr_keys & ex_keys:
    failures.append(f"{len(tr_keys & ex_keys)} compounds are in both the training and external sets")

pred_df = pd.DataFrame(PREDICTIONS)
for name, group in pred_df[pred_df["set"] == "external"].groupby("model"):
    if name not in EXTERNAL:
        continue
    target = name.split(", ")[-1].replace(" (exploratory)", "")
    recomputed = qsar_core.external_metrics(
        model_df[target].to_numpy(dtype=float),
        group["observed"].to_numpy(dtype=float),
        group["predicted"].to_numpy(dtype=float))
    for metric, value in EXTERNAL[name].items():
        if isinstance(value, float) and not np.isclose(value, recomputed[metric], atol=1e-9):
            failures.append(f"{name}: reported {metric} {value} != recomputed {recomputed[metric]}")

for key, blob in RESULTS["models"].items():
    if isinstance(blob, dict) and "r2" in blob and blob["r2"] is not None:
        if not (-5 <= float(blob["r2"]) <= 1.0000001):
            failures.append(f"{key}: r2 {blob['r2']} is outside the possible range")

claimed = {"training compounds": len(model_df), "external compounds": len(ext_df),
           "congeneric series": len(series_list),
           "descriptor columns": len(desc_names),
           "scaffold split test size": SPLIT.report["n_test"]}
for k, v in claimed.items():
    if v is None:
        failures.append(f"{k} has no number behind it")

print("\nassertions:")
print(f"  every endpoint value experimental            {'FAIL' if any('experimental' in f for f in failures) else 'pass'}")
print(f"  training and external disjoint by InChIKey   {'FAIL' if any('both the training' in f for f in failures) else 'pass'}")
print(f"  reported metrics == recomputed from stored predictions  "
      f"{'FAIL' if any('recomputed' in f for f in failures) else 'pass'}"
      f"  ({pred_df['model'].nunique()} models, {len(pred_df)} predictions)")
print(f"  every reported figure has a number behind it {'FAIL' if any('number behind' in f for f in failures) else 'pass'}")
for k, v in claimed.items():
    print(f"    {k:<30}{v}")

if failures:
    for f in failures:
        print("  FAILURE:", f)
    raise AssertionError(f"{len(failures)} verification failure(s); see above")
print("\nall verification checks passed")

RESULTS["environment"] = RUN_ENVIRONMENT
RESULTS["verification"] = {"raw_sha256": raw_hash, "curated_sha256": curated_hash,
                           "chembl_release": release["release"],
                           "predictions_checked": len(pred_df),
                           "models_checked": int(pred_df["model"].nunique()),
                           "failures": failures}

In [ ]:
# Section 15 (summary). Every tier side by side, with the counts that make them comparable
# or not. Read the 'rows' column first: Tiers 1 and 2 are fitted on one congeneric series,
# Tiers 3 and 4 on the whole training paper, which is the plan's question 3 made visible.
summary_rows = []
if TIER1:
    for label, blob in TIER1.items():
        b = blob["best"]
        key = f"Tier 1 Hansch, {label}"
        iv = INTERNAL.get("Tier 1 Hansch, H3 affinity" if "H3" in label
                          else "Tier 1 Hansch, therapeutic index", {})
        summary_rows.append({
            "tier": key, "rows": b.fit.n, "descriptors": b.n_variables,
            "coefficients": b.n_coefficients, "r2": round(b.fit.r2, 4),
            "Q2 LOO": round(iv.get("q2_loo", float("nan")), 4),
            "external": "not applicable (different chemotype)"})
if TIER2.get("model"):
    m = TIER2["model"]
    summary_rows.append({
        "tier": f"Tier 2 Free-Wilson ({m['kind']})", "rows": len(m["y"]),
        "descriptors": len(m["names"]), "coefficients": len(m["names"]),
        "r2": round(m["fit"].r2 if m["fit"] is not None else m["r2"], 4),
        "Q2 LOO": round(m.get("q2_loo", float("nan")), 4),
        "external": "not applicable (different chemotype)"})
for target, blob in TIER3.items():
    iv = INTERNAL.get(f"Tier 3 MLR, {target}", {})
    ex = EXTERNAL.get(f"Tier 3 MLR, {target}", {})
    summary_rows.append({
        "tier": f"Tier 3 MLR, {target}", "rows": blob["fit"].n,
        "descriptors": blob["descriptors"], "coefficients": blob["fit"].k,
        "r2": round(blob["fit"].r2, 4), "Q2 LOO": round(iv.get("q2_loo", float("nan")), 4),
        "external": f"Q2F3 {ex.get('q2_f3', float('nan')):.3f}  CCC {ex.get('ccc', float('nan')):.3f}"})
    pex = EXTERNAL.get(f"Tier 3 PLS, {target}", {})
    summary_rows.append({
        "tier": f"Tier 3 PLS, {target}", "rows": blob["fit"].n,
        "descriptors": f"{len(desc_names)} in, {blob['pls']['n_components']} components",
        "coefficients": blob["pls"]["n_components"], "r2": round(blob["pls"]["r2"], 4),
        "Q2 LOO": float("nan"),
        "external": f"Q2F3 {pex.get('q2_f3', float('nan')):.3f}  CCC {pex.get('ccc', float('nan')):.3f}"})
for target, blob in TIER4.items():
    name = f"Tier 4 {type(blob['model']).__name__}, {target} (exploratory)"
    ex = EXTERNAL.get(name, {})
    summary_rows.append({
        "tier": name, "rows": len(model_df), "descriptors": len(desc_names) - 1,
        "coefficients": "n/a (ensemble)", "r2": round(blob["r2_resubstitution"], 4),
        "Q2 LOO": float("nan"),
        "external": f"Q2F3 {ex.get('q2_f3', float('nan')):.3f}  CCC {ex.get('ccc', float('nan')):.3f}"})

comparison = pd.DataFrame(summary_rows)
print(comparison.to_string(index=False, max_colwidth=46))
RESULTS["comparison"] = summary_rows

print("""
Reading this table honestly:
  - 'descriptors' counts the lipophilicity pair ONCE; 'coefficients' counts both of its terms.
  - Tiers 1 and 2 sit on one congeneric series, so their r2 is not comparable with Tiers 3
    and 4 on the full set, and their blank external column is a property of the Hansch method
    rather than a gap in this run.
  - The external column is another laboratory's compounds, measured on a different H3
    radioligand and reporting hERG as Ki rather than IC50. It is the hardest number here and
    the one that should move an opinion.""")

## §16 Export

The curated table, the per-model coefficient tables, the provenance table, the applicability-domain labels and the fitted models, written to CSV and joblib with one-click download links in Colab.

In [ ]:
# Section 16. Everything worth keeping, written out.
import joblib

OUT = "qsar_outputs"
os.makedirs(OUT, exist_ok=True)
written = []

def write_csv(frame, name):
    path = os.path.join(OUT, name)
    frame.to_csv(path, index=False)
    written.append(path)
    return path

write_csv(pd.DataFrame(table_rows), "curated_dataset.csv")
write_csv(paired, "modelling_table_standardised.csv")
write_csv(provenance, "provenance.csv")
write_csv(pred_df, "predictions.csv")
write_csv(DOMAIN.assign(**{"flag": np.where(DOMAIN["inside_domain"], "",
                                            "OUTSIDE THE APPLICABILITY DOMAIN")}),
          "external_predictions_with_domain.csv")
write_csv(per_doc, "external_metrics_per_document.csv")

coef_rows = []
for label, blob in (TIER1 or {}).items():
    f = blob["best"].fit
    for term, c, se, p in zip(["intercept"] + list(blob["best"].names), f.coef, f.se, f.p):
        coef_rows.append({"model": f"Tier 1 {label}", "term": term, "coefficient": c,
                          "std_error": se, "p": p})
for target, blob in TIER3.items():
    f = blob["fit"]
    for term, c, se, p in zip(["intercept"] + list(blob["ga"].names), f.coef, f.se, f.p):
        coef_rows.append({"model": f"Tier 3 MLR {target}", "term": term, "coefficient": c,
                          "std_error": se, "p": p})
write_csv(pd.DataFrame(coef_rows), "coefficients.csv")

models_path = os.path.join(OUT, "fitted_models.joblib")
joblib.dump({"tier3": {t: {"coef": b["fit"].coef, "terms": b["ga"].names,
                           "columns": b["cols"], "pls": b["pls"]["model"]}
                       for t, b in TIER3.items()},
             "tier4": {t: b["model"] for t, b in TIER4.items()},
             "standardisation": {"mean": mu, "sd": sd, "names": desc_names}},
            models_path)
written.append(models_path)

def json_safe(obj):
    """JSON keys must be strings, and numpy scalars are not JSON. Both appear in RESULTS
    once a pandas frame has been through it, so the whole tree is walked rather than
    patched at the one place it happened to break."""
    if isinstance(obj, dict):
        return {(" / ".join(map(str, k)) if isinstance(k, tuple) else str(k)): json_safe(v)
                for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [json_safe(v) for v in obj.tolist()]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj

results_path = os.path.join(OUT, "results.json")
with open(results_path, "w") as fh:
    json.dump(json_safe(RESULTS), fh, indent=1, default=str)
written.append(results_path)

for name in ("schema_diagram", "workflow_diagram"):
    for ext in (".excalidraw", ".png"):
        if os.path.exists(name + ext):
            os.replace(name + ext, os.path.join(OUT, name + ext))
            written.append(os.path.join(OUT, name + ext))

print("written:")
for p in written:
    print(f"  {p:<58}{os.path.getsize(p):>10,} bytes")

try:
    from google.colab import files           # noqa: F401
    import shutil
    archive = shutil.make_archive("qsar_outputs", "zip", OUT)
    print(f"\nColab detected; downloading {archive}")
    files.download(archive)
except ImportError:
    print(f"\nNot in Colab: the files are in ./{OUT}/")

## §17 How to run this, and what it does not tell you

### Running it

1. Open the notebook in Colab (File → Upload notebook, or open it from GitHub).
2. Runtime → Run all. No API key, no mounted drive, no uploaded file: §3 pulls from the public ChEMBL REST API.
3. Expect roughly ten to twenty minutes, most of it in §3's pull and §10's Optuna search.
4. §16 writes everything to `qsar_outputs/` and, in Colab, offers the zip as a download.
5. Nothing is cached. Re-running re-pulls, and §15 prints the hash of what it got, so a change in ChEMBL between runs is visible rather than silent.

### Declared choices — decisions made on the reader's behalf

| | choice | why |
|---|---|---|
| dataset | H3 / hERG, training CHEMBL1821550 | the search in §3 finds only four documents in the release measuring both endpoints on the same compounds; this is the largest |
| **external set** | **three documents, not one** | **the plan's single external set, CHEMBL1155513, shares 67 of its 75 compounds with the training paper. Used as published it would have measured "external" accuracy mostly on compounds already fitted. The 8 unseen compounds are kept and the other two qualifying documents are added, giving a few dozen compounds from three laboratories, scored per document as well as pooled; §3 prints the exact counts** |
| target variable | `pActivity − pToxicity`, higher is better | one scale, one sign convention, stated in §6 with a worked example |
| **`potential_duplicate`** | **flag recorded, rows kept** | **the plan said drop them; checking showed that would delete 159 of 458 training rows, every one the only row for its compound, while preventing no duplication at all. Deduplication is done on (compound, target, type) instead.** |
| π re-derivation | run against ChEMBL's `alogp`, reported as a consistency probe | `alogp` is itself calculated, so this is not the experimental re-derivation the plan imagined; §6b says so in the output |
| Tier 1 scope | the largest congeneric series; Tiers 3–4 on all training compounds | a Hansch equation is only defined inside a series, so the alternative was to shrink every tier to that series |
| Free–Wilson | penalised if the indicator design is saturated, and labelled as penalised | reporting an OLS fit with more indicators than compounds would be reporting an interpolation |
| Mordred | attempted, RDKit used | Mordred is unmaintained against current Python |
| descriptors | one curated RDKit block, not every available descriptor | a wall of 200 correlated columns would make the genetic algorithm's intercorrelation objective meaningless |

### Limitations

* **The series is small.** Tier 1 fits on a few dozen compounds. The Topliss–Costello ratio is printed next to every fit precisely so this is not hidden.
* **This is not a clinical therapeutic index.** The target variable is a difference of two *in vitro* binding affinities, measured in displacement assays. It carries **no dose, no exposure, no duration and no information about any individual**. It cannot be used to compare drugs in people, to choose between them, or to say anything about safety in use. It ranks analogues inside one chemical series during design, and that is its whole scope.
* **hERG is one liability among many.** A compound with a wide margin here may be unacceptable for reasons this dataset never measured.
* **The external set is small, and it is not the same assay.** A few dozen compounds, from three papers, with four different H3 radioligands between them and three different hERG readouts — one of which is a functional IonWorks patch-clamp IC50 rather than a displacement affinity at all. Expect the external metrics to be pessimistic for reasons that are partly experimental rather than modelling, expect a wide spread between documents, and read the per-document `n` before any per-document metric: at thirteen to twenty compounds, one bad prediction moves Q²F3 visibly.
* **The plan's external set was almost entirely the training set.** That is reported in §5 and §11 rather than corrected quietly, because it is the most instructive thing this run found: an external validation can be built entirely in good faith out of a second citation and still be measuring resubstitution.
* **Tabulated substituent constants carry their own uncertainty**, and transcription is their weakest link. §6b checks what it can and names what it cannot.
* **Tier 4 is exploratory and labelled so.** No machine-learning prediction fills an endpoint anywhere in this notebook.
* **Four of the ten model families the board names are not fitted here** — kernel PLS, SVR, MLP and a graph neural network. §2 prints that list rather than implying the board was covered.
* **One local model is not the global model.** The board's Goal 1 is an assembly of many local models across ATC levels. This is one of them, built to the standard such an assembly would need. The assembly is the next piece of work.

In [ ]:
# Section 17. The run's own footer: what this run actually did, in numbers.
print(f"""
run summary
-----------
ChEMBL release           {release['release']} ({release['release_date']})
training compounds       {len(model_df)} (document {curate.SETS['training']['document']})
external compounds       {len(ext_df)} unseen, from {ext_df['document'].nunique()} documents ({', '.join(sorted(ext_df['document'].unique()))})
congeneric series found  {len(series_list)}; largest n = {main_series.n if main_series else 0}
descriptor columns       {len(desc_names)} ({len(desc_names) - 1} descriptors, the lipophilicity pair counted once)
scaffold split           {SPLIT.report['n_train']} train / {SPLIT.report['n_test']} held out, {SPLIT.report['shared_scaffolds']} shared scaffolds
models fitted            {len(RESULTS['models'])} entries recorded
predictions stored       {len(pred_df)} across {pred_df['model'].nunique()} models
curated table sha256     {curated_hash}
seed                     {SEED}
verification             {'passed' if not RESULTS['verification']['failures'] else 'FAILED'}
""")
print("The target variable is a ratio of two in vitro binding affinities. It is not a dose,")
print("not a clinical therapeutic index, and not advice about any medicine's use in people.")